# Information-theoretic and TDA features for crypto volatility and stress prediction

Curated results notebook for the thesis **Information Theory Methods for Financial Time Series Analysis.** It keeps the cells whose outputs back the thesis tables and removes superseded and exploratory versions.

## Read this first
- **Test period is short.** The holdout is July - September 2025 (about 62 days), a single split. Long horizons (7d, 14d) rest on very few independent observations.
- **W = 96 is the best of four windows on the same holdout**, so it is a post-hoc choice, not a pre-specified one.
- **The headline depends on the asset universe.** On the 8 reported assets the combined configurations improve on the volatility baseline (C: +0.0536, D: +0.0395 at W=192). Re-running the same pipeline with TRX added (9 assets, 54 tasks) gives C: -0.0038 and D: -0.0101 (section Robustness). The notebook also shows the earlier (OLD) feature selection with D: -0.0436.
- **Heterogeneous by asset.** Gains are concentrated in ETH, DOGE, ADA, AVAX; BTC, SOL, BNB are negative or near zero and HAR-RV is better there. The IT-only configuration (B) is negative on average at W=192. In stress classification the Q90 threshold is negative in the main specification (W=192) and only slightly positive at W=96.

## Reproducibility
Saved outputs were produced on the original research dataset and environment. The loader below uses public Binance data (spot close instead of mid-quote, spot high/low), so re-running will give different numbers.
Several cells below also expect objects built in earlier stages that are not included in this file: `X_it_aug`, `X_it_slim`, `X_it_extended`, `X_tda_full`, `tda_A`, `tda_B`, `X_vol`, `ret`, `IT_ALL`, `extra_it_cols`, `extra_tda_cols`, `window_it_features`, and some constants (`ASSETS`, `HORIZONS`, `PURGE_BARS`, `ANN`, `split_ts`). Treat this notebook as a record of results with their code, not as an end-to-end pipeline.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

pd.set_option('display.precision', 8)

## 1. Data
Public Binance 30-minute bars, nine USDT pairs, 2023-01-01 to 2025-09-01. The analysis universe is the 8 assets BTC, ETH, SOL, BNB, XRP, DOGE, ADA, AVAX; TRX is loaded as well and is used only in the robustness check.

In [ ]:
import io, zipfile
import numpy as np
import pandas as pd
import requests

BASE = "https://data.binance.vision/data/spot/monthly/klines"
COLS = ["open_time", "open", "high", "low", "close", "volume", "close_time",
        "quote_volume", "n_trades", "taker_buy_base", "taker_buy_quote", "ignore"]
ASSETS_9 = ("BTC", "ETH", "SOL", "BNB", "XRP", "DOGE", "ADA", "AVAX", "TRX")


def _parse_klines(raw_csv: bytes) -> pd.DataFrame:
    """One Binance kline CSV -> DataFrame; handles optional header and ms/us timestamps."""
    df = pd.read_csv(io.BytesIO(raw_csv), header=None, names=COLS)
    t = pd.to_numeric(df["open_time"], errors="coerce")
    ok = t.notna()
    df, t = df[ok].copy(), t[ok].astype("int64")
    unit = "us" if t.iloc[0] > 10**14 else "ms"
    df.index = pd.to_datetime(t.values, unit=unit)
    return df[["high", "low", "close", "volume", "taker_buy_base"]].astype(float)


def _load_symbol(symbol, start, end, interval="30m"):
    frames = []
    for ym in pd.period_range(start, end, freq="M"):
        url = f"{BASE}/{symbol}/{interval}/{symbol}-{interval}-{ym}.zip"
        r = requests.get(url, timeout=60)
        r.raise_for_status()
        with zipfile.ZipFile(io.BytesIO(r.content)) as z:
            frames.append(_parse_klines(z.read(z.namelist()[0])))
    out = pd.concat(frames).sort_index()
    return out[~out.index.duplicated(keep="first")]


def load_data(assets=ASSETS_9, start="2023-01", end="2025-09", last_bar="2025-09-01 23:30"):
    """Public Binance spot 30-min bars -> the five wide frames used below
    (columns 'BTC-USDT', ...): price_df, volume_ask, volume_bid, high_df, low_df.
    buy volume = taker buy volume, sell volume = total volume - taker buy volume."""
    d = {f"{a}-USDT": _load_symbol(f"{a}USDT", start, end).loc[:last_bar] for a in assets}
    wide = lambda f: pd.DataFrame({k: f(v) for k, v in d.items()})
    return (wide(lambda x: x["close"]),                              # price_df
            wide(lambda x: x["taker_buy_base"]),                     # volume_ask (buy)
            wide(lambda x: x["volume"] - x["taker_buy_base"]),       # volume_bid (sell)
            wide(lambda x: x["high"]), wide(lambda x: x["low"]))

In [ ]:
price_df, volume_ask, volume_bid, high_df, low_df = load_data()

## 2. Feature selection (permutation importance on the training set)
Permutation importance is computed on the training period. The same cell also prints a holdout comparison of feature configurations, in which all mean ΔR² values are slightly negative.

In [ ]:
print("\n" + "=" * 65)
print("  2. PERMUTATION IMPORTANCE ON TRAIN (BTC+ETH, h=4d, logvol)")
print("=" * 65)

perm_results = {}
for pilot_asset in ["BTC", "ETH"]:
    r = ret[pilot_asset]
    vol_fwd = r.rolling(192).std().shift(-192) * ANN
    y_logvol = np.log(vol_fwd.clip(lower=1e-8)).dropna()

    idx = (X_vol.index
           .intersection(X_it_aug.index)
           .intersection(X_tda_full.index)
           .intersection(y_logvol.index))

    valid = (X_vol.loc[idx].notna().all(1) &
             X_it_aug.loc[idx].notna().all(1) &
             X_tda_full.loc[idx].notna().all(1) &
             y_logvol.loc[idx].notna())
    idx = idx[valid]

    tr = idx < (split_ts - pd.Timedelta(minutes=30 * PURGE_BARS))
    idx_tr = idx[tr]

    Xv_tr = X_vol.loc[idx_tr]
    ck = Xv_tr.corrwith(y_logvol.loc[idx_tr]).abs().dropna()
    top3 = ck.nlargest(3).index.tolist()

    mdl_b = _ridge()
    mdl_b.fit(Xv_tr[top3], y_logvol.loc[idx_tr])
    res_tr = y_logvol.loc[idx_tr].values - mdl_b.predict(Xv_tr[top3])

    Xi_all_tr = pd.concat([
        X_it_aug.loc[idx_tr],
        X_tda_full.loc[idx_tr],
    ], axis=1)

    Xi_arr = Xi_all_tr.values
    feat_names = list(Xi_all_tr.columns)

    kf = KFold(n_splits=5, shuffle=False)
    base_cv = float(np.mean(
        cross_val_score(_ridge(), Xi_arr, res_tr, cv=kf, scoring="r2")
    ))

    perm_scores_asset = {}
    for fi, feat in enumerate(feat_names):
        deltas = []
        for _ in range(N_PERM):
            Xp = Xi_arr.copy()
            Xp[:, fi] = RNG.permutation(Xp[:, fi])
            sc = float(np.mean(
                cross_val_score(_ridge(), Xp, res_tr, cv=kf, scoring="r2")
            ))
            deltas.append(sc - base_cv)
        perm_scores_asset[feat] = float(np.mean(deltas))

    perm_results[pilot_asset] = perm_scores_asset
    print(f"  {pilot_asset} baseline CV R² = {base_cv:.4f}")

all_feats = list(perm_results["BTC"].keys())
avg_perm = {
    f: np.mean([perm_results[a].get(f, 0.0) for a in ["BTC", "ETH"]])
    for f in all_feats
}

def categorize(feat):
    if feat in IT_ALL:
        return "IT"
    if feat in TDA_COLS:
        return "TDA"
    return "other"

cat_colors = {
    "IT":    "#2196F3",
    "TDA":   "#FF9800",
    "other": "#9E9E9E",
}

print("\n" + "=" * 65)
print("  3. HOLDOUT CONFIGURATION COMPARISON")
print("=" * 65)

CONFIGS = {
    "slim_IT": IT_SLIM,
    "best_IT": [f for f, d in sorted_perm[:20] if f in IT_ALL],
    "slim+TDA": IT_SLIM + best_tda[:6],
    "best+TDA": [f for f, d in sorted_perm[:16] if f in IT_ALL or f in TDA_COLS],
}

CONFIGS = {k: list(dict.fromkeys(v)) for k, v in CONFIGS.items()}

for name, cols in CONFIGS.items():
    print(f"  {name}: {len(cols)} features")

config_rows = []

for asset in ASSETS:
    r = ret[asset] if asset in ret.columns else None
    if r is None:
        continue

    vol_fwd_h = {}
    for h in HORIZONS:
        vol_fwd_h[h] = r.rolling(h).std().shift(-h) * ANN

    for h, h_label in HORIZONS.items():
        y_raw = np.log(vol_fwd_h[h].clip(lower=1e-8)).dropna()

        for cfg_name, cfg_cols in CONFIGS.items():
            it_cols_used = [c for c in cfg_cols if c in X_it_aug.columns]
            tda_cols_used = [c for c in cfg_cols if c in X_tda_full.columns]

            if it_cols_used or tda_cols_used:
                Xi_cfg = pd.concat(
                    [X_it_aug[it_cols_used]] +
                    ([X_tda_full[tda_cols_used]] if tda_cols_used else []),
                    axis=1
                )
            else:
                Xi_cfg = X_it_aug[IT_SLIM]

            idx = X_vol.index.intersection(Xi_cfg.index).intersection(y_raw.index)
            valid = (X_vol.loc[idx].notna().all(1) &
                     Xi_cfg.loc[idx].notna().all(1) &
                     y_raw.loc[idx].notna())
            idx = idx[valid]

            if len(idx) < 1000:
                continue

            tr = idx < (split_ts - pd.Timedelta(minutes=30 * PURGE_BARS))
            te = idx >= split_ts

            if tr.sum() < 480 or te.sum() < 100:
                continue

            Xv_tr = X_vol.loc[idx[tr]]
            Xv_te = X_vol.loc[idx[te]]
            Xi_tr = Xi_cfg.loc[idx[tr]]
            Xi_te = Xi_cfg.loc[idx[te]]
            ytr = y_raw.loc[idx[tr]]
            yte = y_raw.loc[idx[te]]
            naive = float(ytr.mean())

            ck = Xv_tr.corrwith(ytr).abs().dropna()
            top3 = ck.nlargest(3).index.tolist()

            mb = _ridge()
            mb.fit(Xv_tr[top3], ytr)
            pb_tr = mb.predict(Xv_tr[top3])
            pb_te = mb.predict(Xv_te[top3])

            res_tr = ytr.values - pb_tr

            mc = _ridge()
            mc.fit(Xi_tr, res_tr)
            pc_te = mc.predict(Xi_te)
            pred = pb_te + pc_te

            r2_b = oos_r2(yte.values, pb_te, naive)
            r2_c = oos_r2(yte.values, pred, naive)
            rc = float(pd.Series(pc_te).corr(pd.Series(yte.values - pb_te)))

            config_rows.append({
                "asset": asset,
                "horizon": h_label,
                "h": h,
                "config": cfg_name,
                "r2_base": round(r2_b, 4),
                "r2_cfg": round(r2_c, 4),
                "delta_r2": round(r2_c - r2_b, 4),
                "res_corr": round(rc, 4),
            })

cfg_df = pd.DataFrame(config_rows)

print(f"\n  Mean ΔR² (logvol) by configuration:")
print(f"  {'config':<15} {'mean_ΔR²':>10} {'ΔR²>0':>8} {'mean_rc':>9}")
print("  " + "-" * 45)

for cfg in CONFIGS.keys():
    sub = cfg_df[cfg_df.config == cfg]
    if sub.empty:
        continue
    pos = (sub["delta_r2"] > 0).sum()
    print(f"  {cfg:<15} {sub['delta_r2'].mean():>+10.4f} "
          f"{pos:>5}/{len(sub)} {sub['res_corr'].mean():>+9.4f}")

best_cfg = cfg_df.groupby("config")["delta_r2"].mean().idxmax()
print(f"\n  Best configuration: '{best_cfg}'")

sub_best = cfg_df[cfg_df.config == best_cfg]
print(f"  {'asset':<6} {'h':<4} {'R²_base':>9} {'R²_best':>9} {'ΔR²':>8}")
print("  " + "-" * 40)

prev = None
for _, row in sub_best.sort_values(["asset", "h"]).iterrows():
    if prev and prev != row["asset"]:
        print("  " + "·" * 40)
    prev = row["asset"]
    mark = "↑" if row["delta_r2"] > 0.005 else ("↓" if row["delta_r2"] < -0.005 else "~")
    print(f"  {row['asset']:<6} {row['horizon']:<4}"
          f" {row['r2_base']:>+9.4f} {row['r2_cfg']:>+9.4f}"
          f" {row['delta_r2']:>+8.4f} {mark}")

print("\nBuilding plots...")

fig, axes = plt.subplots(1, 3, figsize=(18, 7))

ax = axes[0]
feats_sorted = [f for f, _ in sorted_perm]
vals_sorted = [avg_perm[f] for f in feats_sorted]
colors_bar = [cat_colors.get(categorize(f), "#9E9E9E") for f in feats_sorted]
y_pos = np.arange(len(feats_sorted))

ax.barh(y_pos, vals_sorted, color=colors_bar, alpha=0.85, height=0.8)
ax.axvline(0, color="black", lw=0.8)
ax.axvline(-0.002, color="green", lw=0.8, ls=":", alpha=0.6)
ax.axvline(+0.002, color="red", lw=0.8, ls=":", alpha=0.6)
ax.set_yticks(y_pos)
ax.set_yticklabels(feats_sorted, fontsize=7)
ax.set_xlabel("Δ CV R² under permutation\n(< 0 = feature is useful)")
ax.set_title("Permutation importance\n(BTC+ETH, h=4d, logvol)", fontsize=10)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

from matplotlib.patches import Patch

legend_elems = [
    Patch(facecolor=cat_colors["IT"], label="IT features", alpha=0.85),
    Patch(facecolor=cat_colors["TDA"], label="TDA features", alpha=0.85),
]

ax.legend(handles=legend_elems, fontsize=8, loc="lower right", frameon=True)

ax = axes[1]
cfg_order = list(CONFIGS.keys())
asset_order = ASSETS
x = np.arange(len(asset_order))
w = 0.18
cfg_colors = ["#90CAF9", "#2196F3", "#4CAF50", "#FF9800"]

for ci, cfg in enumerate(cfg_order):
    sub = cfg_df[cfg_df.config == cfg]
    vals = []
    for asset in asset_order:
        v = sub[sub.asset == asset]["delta_r2"].mean()
        vals.append(v if not np.isnan(v) else 0.0)

    ax.bar(
        x + (ci - 1.5) * w,
        vals,
        w * 0.9,
        label=cfg,
        color=cfg_colors[ci],
        alpha=0.85
    )

ax.axhline(0, color="black", lw=0.8)
ax.set_xticks(x)
ax.set_xticklabels(asset_order)
ax.set_ylabel("Mean ΔR² (all horizons)")
ax.set_title("ΔR² by configuration × asset\n(logvol, holdout 2025)", fontsize=10)
ax.legend(fontsize=8)
ax.grid(alpha=0.2, axis="y")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax = axes[2]
cat_corrs = {}
for feat, c in {**it_corrs, **tda_corrs}.items():
    cat = categorize(feat)
    cat_corrs.setdefault(cat, []).append(c)

cats_to_plot = [
    "IT_slim", "IT_base_other", "IT_extra",
    "TDA_A", "TDA_BI", "TDA_BII", "TDA_BC", "TDA_extra"
]
data_plot = [cat_corrs.get(c, [0]) for c in cats_to_plot]

bp = ax.boxplot(
    data_plot,
    labels=cats_to_plot,
    patch_artist=True,
    medianprops=dict(color="black", lw=1.5),
    flierprops=dict(marker=".", markersize=4, alpha=0.5)
)

for patch, cat in zip(bp["boxes"], cats_to_plot):
    patch.set_facecolor(cat_colors.get(cat, "#9E9E9E"))
    patch.set_alpha(0.75)

ax.axhline(0, color="black", lw=0.5)
ax.set_ylabel("corr(feature, vol_BTC) on test period")
ax.set_title("Feature correlation with vol_BTC\nby category", fontsize=10)
ax.set_xticklabels(cats_to_plot, rotation=20, ha="right", fontsize=8)
ax.grid(alpha=0.2, axis="y")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.suptitle("Feature utility analysis: IT + TDA with alternative MI estimators",
             fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig("fig_feature_analysis_extended.png", bbox_inches="tight", dpi=150)
plt.savefig("fig_feature_analysis_extended.pdf", bbox_inches="tight", dpi=150)
plt.show()
print("  Saved: fig_feature_analysis_extended.png / .pdf")

print(f"\n{'=' * 65}")
print("  FINAL: best feature set")
print(f"{'=' * 65}")

print(f"\n  Configuration '{best_cfg}' ({len(CONFIGS[best_cfg])} features):")
for f in CONFIGS[best_cfg]:
    cat = categorize(f)
    imp = avg_perm.get(f, 0.0)
    print(f"    {f:<36} [{cat}]  perm={imp:>+.4f}")

print(f"\n  Comparison of all configurations:")
for cfg in cfg_order:
    sub = cfg_df[cfg_df.config == cfg]
    print(f"    {cfg:<15}: ΔR²={sub['delta_r2'].mean():>+.4f}  "
          f"rc={sub['res_corr'].mean():>+.4f}  "
          f"↑{(sub['delta_r2'] > 0).sum()}/{len(sub)}")

feat_analysis = {
    "perm": avg_perm,
    "cfg_df": cfg_df,
    "it_corrs": it_corrs,
    "tda_corrs": tda_corrs,
    "best_cfg": best_cfg,
    "best_cols": CONFIGS[best_cfg],
    "X_it_aug": X_it_aug,
    "X_tda_full": X_tda_full,
    "extra_it_cols": extra_it_cols,
    "extra_tda_cols": extra_tda_cols,
}
print(f"\nfeat_analysis saved in memory")


  2. PERMUTATION IMPORTANCE ON TRAIN (BTC+ETH, h=4d, logvol)
  BTC baseline CV R² = -1.0890
  ETH baseline CV R² = -1.4195

  3. HOLDOUT CONFIGURATION COMPARISON
  slim_IT: 10 features
  best_IT: 10 features
  slim+TDA: 16 features
  best+TDA: 16 features

  Mean ΔR² (logvol) by configuration:
  config            mean_ΔR²    ΔR²>0   mean_rc
  ---------------------------------------------
  slim_IT            -0.0210     4/12   +0.1162
  best_IT            -0.0385     6/12   +0.0333
  slim+TDA           -0.0036     5/12   +0.0870
  best+TDA           -0.0251     5/12   +0.0660

  Best configuration: 'slim+TDA'
  asset  h      R²_base   R²_best      ΔR²
  ----------------------------------------
  BNB    2d     +0.3104   +0.2663  -0.0441 ↓
  BNB    4d     +0.2152   +0.1233  -0.0919 ↓
  ········································
  BTC    2d     +0.1300   +0.0053  -0.1247 ↓
  BTC    4d     -0.1246   -0.2650  -0.1404 ↓
  ········································
  DOGE   2d     +0.2001   +0.3

## 3. Volatility regression, main specification (W = 192)
Source of thesis Table 1 and the 32/48, 21/48, 31/48 hit rates.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from scipy import stats as sp_stats

from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

SPLIT_DATE = "2025-07-01"
PURGE_BARS = 96

HORIZONS = {
    48:  "1d",
    96:  "2d",
    192: "4d",
    288: "6d",
    336: "7d",
    672: "14d",
}

ASSETS = [a for a in ["BTC","ETH","SOL","BNB","XRP","DOGE","ADA","AVAX"] if a in ret.columns]

VOL_WINDOW = 48
ANN = np.sqrt(365 * 48)

PERM_USEFUL_THRESHOLD = -0.002

TOP_IT = 10
TOP_TDA = 6

def _ridge():
    return Pipeline([
        ("sc", StandardScaler()),
        ("m", RidgeCV(alphas=[0.01, 0.1, 1, 10, 100]))
    ])

def oos_r2(y_true, y_pred, naive_val):
    mse_m = np.mean((y_true - y_pred) ** 2)
    mse_n = np.mean((y_true - naive_val) ** 2)
    return float(1.0 - mse_m / mse_n) if mse_n > 1e-12 else np.nan

def res_corr(residuals_true, residuals_pred):
    return float(pd.Series(residuals_pred).corr(pd.Series(residuals_true)))

def hit_rate_vs_baseline(y_true, pred_base, pred_model):
    e_base = (y_true - pred_base) ** 2
    e_mod  = (y_true - pred_model) ** 2
    return float(np.mean(e_mod < e_base))

def dm_test(loss_diff):
    n = len(loss_diff)
    m = loss_diff.mean()
    s = loss_diff.std(ddof=1)
    t = m / (s / np.sqrt(n)) if s > 1e-12 else 0.0
    p = float(sp_stats.t.sf(abs(t), df=n - 1) * 2) if n > 1 else np.nan
    return float(t), float(p)

def sig(p):
    if not np.isfinite(p):
        return ""
    if p < 0.01:
        return "***"
    if p < 0.05:
        return "**"
    if p < 0.10:
        return "*"
    return ""

if "feat_analysis" in dir():
    avg_perm = feat_analysis["perm"]

    if "X_it_aug" in feat_analysis:
        X_it_all = feat_analysis["X_it_aug"].copy()
    else:
        X_it_all = X_it_slim.copy()

    if "X_tda_full" in feat_analysis:
        X_tda_all = feat_analysis["X_tda_full"].copy()
    else:
        raise ValueError("feat_analysis exists, but X_tda_full is missing inside it.")
else:
    if "avg_perm" not in dir():
        raise ValueError("avg_perm / feat_analysis['perm'] not found in memory.")
    if "X_it_aug" not in dir():
        raise ValueError("X_it_aug not found in memory.")
    if "X_tda_full" not in dir():
        raise ValueError("X_tda_full not found in memory.")

    X_it_all = X_it_aug.copy()
    X_tda_all = X_tda_full.copy()

perm_series = pd.Series(avg_perm).sort_values()

print("=" * 80)
print("NEW HOLDOUT EXPERIMENT WITH THE BEST FEATURES FROM PERMUTATION IMPORTANCE")
print("=" * 80)

it_candidates = [
    f for f in perm_series.index
    if (f in X_it_all.columns) and (perm_series[f] < PERM_USEFUL_THRESHOLD)
]

tda_candidates = [
    f for f in perm_series.index
    if (f in X_tda_all.columns) and (perm_series[f] < PERM_USEFUL_THRESHOLD)
]

BEST_IT = it_candidates[:TOP_IT]
BEST_TDA = tda_candidates[:TOP_TDA]

print("\nSelected IT features:")
for f in BEST_IT:
    print(f"  {f:<36} perm={perm_series[f]:+.4f}")

print("\nSelected TDA features:")
for f in BEST_TDA:
    print(f"  {f:<36} perm={perm_series[f]:+.4f}")

if len(BEST_IT) == 0:
    raise ValueError("No useful IT features survived the threshold.")
if len(BEST_TDA) == 0:
    raise ValueError("No useful TDA features survived the threshold.")

print(f"\nIT count:  {len(BEST_IT)}")
print(f"TDA count: {len(BEST_TDA)}")

split_ts = pd.Timestamp(SPLIT_DATE)
rows = []

for asset in ASSETS:
    r = ret[asset]

    for h, h_label in HORIZONS.items():
        vol_fwd = r.rolling(h).std().shift(-h) * ANN
        y_raw = np.log(vol_fwd.clip(lower=1e-8)).dropna()

        Xi_it = X_it_all[BEST_IT].copy()
        Xi_tda = X_tda_all[BEST_TDA].copy()

        idx = (
            X_vol.index
            .intersection(Xi_it.index)
            .intersection(Xi_tda.index)
            .intersection(y_raw.index)
        )

        valid = (
            X_vol.loc[idx].notna().all(1)
            & Xi_it.loc[idx].notna().all(1)
            & Xi_tda.loc[idx].notna().all(1)
            & y_raw.loc[idx].notna()
        )
        idx = idx[valid]

        if len(idx) < 1000:
            continue

        tr = idx < (split_ts - pd.Timedelta(minutes=30 * PURGE_BARS))
        te = idx >= split_ts

        if tr.sum() < 480 or te.sum() < 100:
            continue

        Xv_tr = X_vol.loc[idx[tr]]
        Xv_te = X_vol.loc[idx[te]]

        Xit_tr = Xi_it.loc[idx[tr]]
        Xit_te = Xi_it.loc[idx[te]]

        Xtd_tr = Xi_tda.loc[idx[tr]]
        Xtd_te = Xi_tda.loc[idx[te]]

        ytr = y_raw.loc[idx[tr]]
        yte = y_raw.loc[idx[te]]

        naive = float(ytr.mean())

        ck = Xv_tr.corrwith(ytr).abs().replace([np.inf, -np.inf], np.nan).dropna()
        top3 = ck.nlargest(3).index.tolist()

        mb = _ridge()
        mb.fit(Xv_tr[top3], ytr)

        pb_tr = mb.predict(Xv_tr[top3])
        pb_te = mb.predict(Xv_te[top3])

        res_tr = ytr.values - pb_tr
        res_te = yte.values - pb_te

        mit = _ridge()
        mit.fit(Xit_tr, res_tr)
        pit_te = mit.predict(Xit_te)
        pred_it = pb_te + pit_te

        mtda = _ridge()
        mtda.fit(Xtd_tr, res_tr)
        ptda_te = mtda.predict(Xtd_te)
        pred_tda = pb_te + ptda_te

        Xc_tr = pd.concat([Xit_tr, Xtd_tr], axis=1)
        Xc_te = pd.concat([Xit_te, Xtd_te], axis=1)

        mct = _ridge()
        mct.fit(Xc_tr, res_tr)
        pct_te = mct.predict(Xc_te)
        pred_all = pb_te + pct_te

        r2_A = oos_r2(yte.values, pb_te, naive)
        r2_B = oos_r2(yte.values, pred_it, naive)
        r2_C = oos_r2(yte.values, pred_tda, naive)
        r2_D = oos_r2(yte.values, pred_all, naive)

        rc_B = res_corr(res_te, pit_te)
        rc_C = res_corr(res_te, ptda_te)
        rc_D = res_corr(res_te, pct_te)

        mse_A = np.mean((yte.values - pb_te) ** 2)
        mse_B = np.mean((yte.values - pred_it) ** 2)
        mse_C = np.mean((yte.values - pred_tda) ** 2)
        mse_D = np.mean((yte.values - pred_all) ** 2)

        hr_B = hit_rate_vs_baseline(yte.values, pb_te, pred_it)
        hr_C = hit_rate_vs_baseline(yte.values, pb_te, pred_tda)
        hr_D = hit_rate_vs_baseline(yte.values, pb_te, pred_all)

        dB = (yte.values - pb_te) ** 2 - (yte.values - pred_it) ** 2
        dC = (yte.values - pb_te) ** 2 - (yte.values - pred_tda) ** 2
        dD = (yte.values - pb_te) ** 2 - (yte.values - pred_all) ** 2

        _, p_B = dm_test(dB)
        _, p_C = dm_test(dC)
        _, p_D = dm_test(dD)

        rows.append({
            "asset": asset,
            "horizon": h_label,
            "h": h,

            "r2_A": round(r2_A, 4),
            "r2_B": round(r2_B, 4),
            "r2_C": round(r2_C, 4),
            "r2_D": round(r2_D, 4),

            "dR2_B": round(r2_B - r2_A, 4),
            "dR2_C": round(r2_C - r2_A, 4),
            "dR2_D": round(r2_D - r2_A, 4),

            "rc_B": round(rc_B, 4),
            "rc_C": round(rc_C, 4),
            "rc_D": round(rc_D, 4),

            "mse_ratio_B": round(mse_B / mse_A, 4) if mse_A > 0 else np.nan,
            "mse_ratio_C": round(mse_C / mse_A, 4) if mse_A > 0 else np.nan,
            "mse_ratio_D": round(mse_D / mse_A, 4) if mse_A > 0 else np.nan,

            "hit_B": round(hr_B, 4),
            "hit_C": round(hr_C, 4),
            "hit_D": round(hr_D, 4),

            "p_B": round(p_B, 4),
            "p_C": round(p_C, 4),
            "p_D": round(p_D, 4),

            "sig_B": sig(p_B),
            "sig_C": sig(p_C),
            "sig_D": sig(p_D),

            "n_train": int(tr.sum()),
            "n_test": int(te.sum()),
        })

df_best = pd.DataFrame(rows)
print(f"\nRows collected: {len(df_best)}")

if df_best.empty:
    raise ValueError("The result dataframe is empty. Check intersections, NaNs, and sample sizes.")

print("\n" + "=" * 100)
print("FINAL COMPARISON WITH AUTOMATICALLY SELECTED BEST FEATURES")
print("Target: logvol = log(realized_vol_fwd_h)")
print("A = Baseline(vol) | B = +best IT | C = +best TDA | D = +best IT + best TDA")
print("=" * 100)

print(
    f"\n  {'asset':<6} {'h':<4}"
    f" {'R²_A':>9} {'R²_B':>9} {'R²_C':>9} {'R²_D':>9}"
    f" {'ΔB':>8} {'ΔC':>8} {'ΔD':>8}  best"
)
print("  " + "-" * 90)

prev = None
for _, row in df_best.sort_values(["asset", "h"]).iterrows():
    if prev and prev != row["asset"]:
        print("  " + "·" * 90)
    prev = row["asset"]

    vals = {"A": row["r2_A"], "B": row["r2_B"], "C": row["r2_C"], "D": row["r2_D"]}
    best = max(vals, key=vals.get)

    mb = "↑" if row["dR2_B"] > 0.005 else ("↓" if row["dR2_B"] < -0.005 else "~")
    mc = "↑" if row["dR2_C"] > 0.005 else ("↓" if row["dR2_C"] < -0.005 else "~")
    md = "↑" if row["dR2_D"] > 0.005 else ("↓" if row["dR2_D"] < -0.005 else "~")

    print(
        f"  {row['asset']:<6} {row['horizon']:<4}"
        f" {row['r2_A']:>+9.4f}"
        f" {row['r2_B']:>+9.4f}"
        f" {row['r2_C']:>+9.4f}"
        f" {row['r2_D']:>+9.4f}"
        f" {row['dR2_B']:>+7.4f}{mb}"
        f" {row['dR2_C']:>+7.4f}{mc}"
        f" {row['dR2_D']:>+7.4f}{md}"
        f"  {best}"
    )

print("\n" + "=" * 75)
print("SUMMARY")
print("=" * 75)

n = len(df_best)
print(f"""
  Config         mean ΔR²   ΔR²>0    mean rc   MSE<1    hit rate
  ----------------------------------------------------------------
  B (+best IT)   {df_best['dR2_B'].mean():>+8.4f}   {(df_best['dR2_B']>0).sum():>2}/{n}   {df_best['rc_B'].mean():>+7.4f}   {(df_best['mse_ratio_B']<1).sum():>2}/{n}   {df_best['hit_B'].mean():.3f}
  C (+best TDA)  {df_best['dR2_C'].mean():>+8.4f}   {(df_best['dR2_C']>0).sum():>2}/{n}   {df_best['rc_C'].mean():>+7.4f}   {(df_best['mse_ratio_C']<1).sum():>2}/{n}   {df_best['hit_C'].mean():.3f}
  D (+both)      {df_best['dR2_D'].mean():>+8.4f}   {(df_best['dR2_D']>0).sum():>2}/{n}   {df_best['rc_D'].mean():>+7.4f}   {(df_best['mse_ratio_D']<1).sum():>2}/{n}   {df_best['hit_D'].mean():.3f}
""")

print("Top-5 tasks by ΔR² for configuration D:")
for _, row in df_best.nlargest(5, "dR2_D").iterrows():
    print(
        f"  {row['asset']} h={row['horizon']}: "
        f"R²_A={row['r2_A']:+.4f} -> "
        f"R²_B={row['r2_B']:+.4f} -> "
        f"R²_C={row['r2_C']:+.4f} -> "
        f"R²_D={row['r2_D']:+.4f} "
        f"(ΔD={row['dR2_D']:+.4f}, p={row['p_D']:.4f}{row['sig_D']})"
    )

asset_summary = (
    df_best.groupby("asset", as_index=False)
    .agg(
        r2_A=("r2_A", "mean"),
        r2_B=("r2_B", "mean"),
        r2_C=("r2_C", "mean"),
        r2_D=("r2_D", "mean"),

        dR2_B=("dR2_B", "mean"),
        dR2_C=("dR2_C", "mean"),
        dR2_D=("dR2_D", "mean"),

        rc_B=("rc_B", "mean"),
        rc_C=("rc_C", "mean"),
        rc_D=("rc_D", "mean"),

        mse_ratio_B=("mse_ratio_B", "mean"),
        mse_ratio_C=("mse_ratio_C", "mean"),
        mse_ratio_D=("mse_ratio_D", "mean"),

        hit_B=("hit_B", "mean"),
        hit_C=("hit_C", "mean"),
        hit_D=("hit_D", "mean"),
    )
)

def choose_best_config(row):
    vals = {
        "A": row["r2_A"],
        "B": row["r2_B"],
        "C": row["r2_C"],
        "D": row["r2_D"],
    }
    return max(vals, key=vals.get)

def choose_best_addon(row):
    vals = {
        "B": row["dR2_B"],
        "C": row["dR2_C"],
        "D": row["dR2_D"],
    }
    return max(vals, key=vals.get)

asset_summary["best_config"] = asset_summary.apply(choose_best_config, axis=1)
asset_summary["best_addon"] = asset_summary.apply(choose_best_addon, axis=1)

asset_summary = asset_summary.sort_values(
    ["r2_D", "r2_C", "r2_B", "r2_A"],
    ascending=False
).reset_index(drop=True)

print("\n" + "=" * 90)
print("ASSET-LEVEL AGGREGATION (MEAN OVER HORIZONS)")
print("=" * 90)

print(
    f"\n  {'asset':<6}"
    f" {'R²_A':>10} {'R²_B':>10} {'R²_C':>10} {'R²_D':>10}"
    f" {'ΔB':>9} {'ΔC':>9} {'ΔD':>9}"
    f" {'best_cfg':>10} {'best_addon':>12}"
)

print("  " + "-" * 100)

for _, row in asset_summary.iterrows():
    print(
        f"  {row['asset']:<6}"
        f" {row['r2_A']:>+10.4f}"
        f" {row['r2_B']:>+10.4f}"
        f" {row['r2_C']:>+10.4f}"
        f" {row['r2_D']:>+10.4f}"
        f" {row['dR2_B']:>+9.4f}"
        f" {row['dR2_C']:>+9.4f}"
        f" {row['dR2_D']:>+9.4f}"
        f" {row['best_config']:>10}"
        f" {row['best_addon']:>12}"
    )

print("\nBest configuration counts across assets:")
print(asset_summary["best_config"].value_counts().sort_index())

print("\nBest incremental addon counts across assets:")
print(asset_summary["best_addon"].value_counts().sort_index())

print("\nBuilding final plots...")

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

asset_order = (
    df_best.groupby("asset")["r2_D"]
    .mean()
    .sort_values(ascending=False)
    .index
    .tolist()
)

h_order = ["1d", "2d", "4d", "6d", "7d", "14d"]
x = np.arange(len(asset_order))
w = 0.2

ax = axes[0, 0]
for ci, (col, label, color) in enumerate([
    ("r2_A", "A: Baseline",   "#9E9E9E"),
    ("r2_B", "B: +best IT",   "#2196F3"),
    ("r2_C", "C: +best TDA",  "#FF9800"),
    ("r2_D", "D: +best both", "#4CAF50"),
]):
    vals = [df_best[df_best.asset == a][col].mean() for a in asset_order]
    ax.bar(x + (ci - 1.5) * w, vals, w * 0.9, label=label, color=color, alpha=0.85)

ax.axhline(0, color="black", lw=0.8)
ax.set_xticks(x)
ax.set_xticklabels(asset_order, fontsize=9)
ax.set_ylabel("OOS R² (vs naive mean logvol)")
ax.set_title("OOS R² by asset — 4 configurations\n(logvol, all horizons)", fontsize=10)
ax.legend(fontsize=8)
ax.grid(alpha=0.2, axis="y")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax = axes[0, 1]
for col, label, color, ls in [
    ("r2_A", "A: Baseline",   "#9E9E9E", "-"),
    ("r2_B", "B: +best IT",   "#2196F3", "-"),
    ("r2_C", "C: +best TDA",  "#FF9800", "--"),
    ("r2_D", "D: +best both", "#4CAF50", "-"),
]:
    vals = [df_best[df_best.horizon == h][col].mean() for h in h_order]
    ax.plot(h_order, vals, "o" + ls, color=color, lw=2, label=label, ms=7)

ax.axhline(0, color="black", lw=0.5, ls=":")
ax.set_ylabel("OOS R²")
ax.set_xlabel("Forecast horizon")
ax.set_title("OOS R² by horizon\n(mean over assets)", fontsize=10)
ax.legend(fontsize=8)
ax.grid(alpha=0.2)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax = axes[1, 0]
x2 = np.arange(len(asset_order))
w2 = 0.28

for ci, (col, label, color) in enumerate([
    ("dR2_B", "B: +best IT",   "#2196F3"),
    ("dR2_C", "C: +best TDA",  "#FF9800"),
    ("dR2_D", "D: +best both", "#4CAF50"),
]):
    vals = [df_best[df_best.asset == a][col].mean() for a in asset_order]
    ax.bar(x2 + (ci - 1) * w2, vals, w2 * 0.9, label=label, color=color, alpha=0.85)

ax.axhline(0, color="black", lw=0.8)
ax.set_xticks(x2)
ax.set_xticklabels(asset_order, fontsize=9)
ax.set_ylabel("ΔR² relative to baseline A")
ax.set_title("Incremental ΔR² over baseline\n(logvol, all horizons)", fontsize=10)
ax.legend(fontsize=8)
ax.grid(alpha=0.2, axis="y")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax = axes[1, 1]
pivot = df_best.pivot_table(
    index="asset",
    columns="horizon",
    values="mse_ratio_D",
    aggfunc="mean"
)
pivot = pivot.reindex(asset_order)[h_order]

im = ax.imshow(pivot.values, cmap="RdYlGn_r", vmin=0.85, vmax=1.15, aspect="auto")
ax.set_xticks(range(len(h_order)))
ax.set_xticklabels(h_order)
ax.set_yticks(range(len(asset_order)))
ax.set_yticklabels(asset_order)

for i in range(len(asset_order)):
    for j in range(len(h_order)):
        val = pivot.values[i, j]
        if np.isfinite(val):
            col = "white" if abs(val - 1) > 0.07 else "black"
            mark = "✓" if val < 1.0 else ""
            ax.text(j, i, f"{val:.3f}{mark}", ha="center", va="center", fontsize=8, color=col)

plt.colorbar(im, ax=ax, label="MSE ratio (D / baseline)\n<1.0 = better ✓")
ax.set_title("MSE ratio for configuration D\n(green < 1 = better than baseline)", fontsize=10)

plt.suptitle(
    "Final comparison with automatically selected best features\n"
    f"Target: logvol = log(realized_vol_fwd) | holdout {SPLIT_DATE} ->",
    fontsize=12,
    y=1.01
)
plt.tight_layout()
plt.savefig("fig_final_comparison_best_features.png", bbox_inches="tight", dpi=150)
plt.savefig("fig_final_comparison_best_features.pdf", bbox_inches="tight", dpi=150)
plt.show()

print("Saved: fig_final_comparison_best_features.png / .pdf")

final_comparison_best = df_best.copy()
best_feature_sets = {
    "BEST_IT": BEST_IT,
    "BEST_TDA": BEST_TDA,
}
print(f"\nfinal_comparison_best saved in memory ({len(df_best)} rows)")
print("best_feature_sets saved in memory")

NEW HOLDOUT EXPERIMENT WITH THE BEST FEATURES FROM PERMUTATION IMPORTANCE

Selected IT features:
  kl_BTC                               perm=-0.0811
  coinfo4_std                          perm=-0.0297
  kl_ADA                               perm=-0.0293
  nmi_lag1_mean                        perm=-0.0283
  coinfo3_std                          perm=-0.0216
  pairwise_mi_mean                     perm=-0.0064

Selected TDA features:
  BI_L1_H0                             perm=-0.1617
  BC_ent_H1                            perm=-0.0687
  BII_tp_H1                            perm=-0.0671
  BII_L1_H1                            perm=-0.0454
  A_T_H0                               perm=-0.0196
  A_E_H0                               perm=-0.0115

IT count:  6
TDA count: 6

Rows collected: 48

FINAL COMPARISON WITH AUTOMATICALLY SELECTED BEST FEATURES
Target: logvol = log(realized_vol_fwd_h)
A = Baseline(vol) | B = +best IT | C = +best TDA | D = +best IT + best TDA

  asset  h         R²_A      R²

## 4. Robustness: earlier selection (OLD) and the 9-asset universe including TRX
On the 48 common tasks D goes from -0.0436 (OLD) to +0.0395 (NEW); on all 54 tasks including TRX, C and D are about zero or negative.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

old_rows = [
    ("ADA","1d",  0.2670,  0.2523,  0.3067,  0.2982),
    ("ADA","2d",  0.2413,  0.2165,  0.2940,  0.2668),
    ("ADA","4d",  0.1948,  0.1208,  0.2548,  0.1489),
    ("ADA","6d",  0.1330,  0.0360,  0.1957,  0.0616),
    ("ADA","7d",  0.1210,  0.0048,  0.1891,  0.0320),
    ("ADA","14d", -0.0990, -0.5945,  0.0657, -0.4020),

    ("AVAX","1d",  0.1260,  0.0734,  0.1560,  0.1200),
    ("AVAX","2d",  0.1056, -0.0078,  0.1448,  0.0649),
    ("AVAX","4d",  0.1883, -0.0036,  0.2274,  0.0537),
    ("AVAX","6d",  0.0701, -0.0685,  0.1045, -0.0276),
    ("AVAX","7d",  0.0419, -0.0664,  0.0856, -0.0148),
    ("AVAX","14d", -0.5613, -1.5173, -0.4966, -1.2211),

    ("BNB","1d",  0.3898,  0.3507,  0.4061,  0.3666),
    ("BNB","2d",  0.3814,  0.3232,  0.3986,  0.3422),
    ("BNB","4d",  0.3750,  0.2912,  0.3801,  0.2993),
    ("BNB","6d",  0.3524,  0.2285,  0.3478,  0.2348),
    ("BNB","7d",  0.3428,  0.1913,  0.3343,  0.2024),
    ("BNB","14d", 0.1255, -0.8769,  0.1369, -0.8856),

    ("BTC","1d",  0.3555,  0.3274,  0.3543,  0.3252),
    ("BTC","2d",  0.3453,  0.3056,  0.3431,  0.3086),
    ("BTC","4d",  0.3952,  0.3477,  0.3926,  0.3694),
    ("BTC","6d",  0.4248,  0.3915,  0.4349,  0.4361),
    ("BTC","7d",  0.4664,  0.4462,  0.4815,  0.4962),
    ("BTC","14d", 0.2806,  0.2824,  0.3259,  0.4183),

    ("DOGE","1d",  0.1887,  0.1982,  0.2268,  0.2713),
    ("DOGE","2d",  0.2228,  0.2388,  0.2771,  0.3363),
    ("DOGE","4d",  0.2263,  0.2257,  0.2890,  0.3211),
    ("DOGE","6d",  0.1637,  0.1805,  0.2222,  0.2612),
    ("DOGE","7d",  0.1713,  0.2170,  0.2361,  0.3022),
    ("DOGE","14d", -0.0064, 0.0727,  0.1403,  0.2128),

    ("ETH","1d",  0.1864,  0.2211,  0.2005,  0.2662),
    ("ETH","2d",  0.2248,  0.2558,  0.2311,  0.3096),
    ("ETH","4d",  0.2535,  0.2352,  0.2300,  0.2540),
    ("ETH","6d",  0.2426,  0.2630,  0.2084,  0.2561),
    ("ETH","7d",  0.2724,  0.3207,  0.2348,  0.3097),
    ("ETH","14d", 0.1574,  0.2398,  0.1866,  0.2865),

    ("SOL","1d",  0.2840,  0.2522,  0.2735,  0.2482),
    ("SOL","2d",  0.2923,  0.2386,  0.2725,  0.2349),
    ("SOL","4d",  0.3340,  0.2751,  0.2914,  0.2573),
    ("SOL","6d",  0.3272,  0.2905,  0.2512,  0.2543),
    ("SOL","7d",  0.3448,  0.3156,  0.2556,  0.2813),
    ("SOL","14d", 0.3746,  0.1921,  0.1266,  0.1734),

    ("XRP","1d",  0.1001,  0.0789,  0.1249,  0.1148),
    ("XRP","2d",  0.0358,  0.0298,  0.0795,  0.0777),
    ("XRP","4d", -0.0082,  0.0007,  0.0564,  0.0185),
    ("XRP","6d", -0.0475, -0.0402,  0.0159, -0.0432),
    ("XRP","7d", -0.0936, -0.0814, -0.0269, -0.0813),
    ("XRP","14d",-0.2833, -0.4040, -0.1395, -0.2821),
]

new_rows = [
    ("ADA","1d",  0.2670,  0.2627,  0.3490,  0.3477),
    ("ADA","2d",  0.2413,  0.2377,  0.3515,  0.3568),
    ("ADA","4d",  0.1948,  0.1633,  0.3151,  0.3163),
    ("ADA","6d",  0.1330,  0.0759,  0.2573,  0.2405),
    ("ADA","7d",  0.1210,  0.0510,  0.2543,  0.2194),
    ("ADA","14d", -0.0990, -0.3930,  0.1013,  0.0318),

    ("AVAX","1d",  0.1260,  0.1032,  0.1651,  0.1389),
    ("AVAX","2d",  0.1056,  0.0853,  0.1696,  0.1365),
    ("AVAX","4d",  0.1883,  0.1486,  0.2247,  0.1632),
    ("AVAX","6d",  0.0701,  0.0207,  0.1226,  0.0730),
    ("AVAX","7d",  0.0419, -0.0111,  0.0936,  0.0388),
    ("AVAX","14d", -0.5613, -1.2264, -0.5189, -0.7516),

    ("BNB","1d",  0.3898,  0.3981,  0.3883,  0.4130),
    ("BNB","2d",  0.3814,  0.4046,  0.3624,  0.4173),
    ("BNB","4d",  0.3750,  0.3907,  0.3010,  0.3643),
    ("BNB","6d",  0.3524,  0.3469,  0.2511,  0.3121),
    ("BNB","7d",  0.3428,  0.3181,  0.2339,  0.2869),
    ("BNB","14d", 0.1255, -0.5463, -0.4452, -0.6457),

    ("BTC","1d",  0.3555,  0.3069,  0.2920,  0.3037),
    ("BTC","2d",  0.3453,  0.2945,  0.2805,  0.3172),
    ("BTC","4d",  0.3952,  0.3511,  0.3619,  0.3789),
    ("BTC","6d",  0.4248,  0.3936,  0.4057,  0.4196),
    ("BTC","7d",  0.4664,  0.4395,  0.4509,  0.4487),
    ("BTC","14d", 0.2806,  0.2897,  0.4717,  0.4211),

    ("DOGE","1d",  0.1887,  0.2262,  0.3054,  0.3055),
    ("DOGE","2d",  0.2228,  0.2926,  0.3905,  0.4061),
    ("DOGE","4d",  0.2263,  0.2990,  0.4261,  0.4243),
    ("DOGE","6d",  0.1637,  0.2374,  0.3269,  0.3319),
    ("DOGE","7d",  0.1713,  0.2516,  0.3026,  0.3069),
    ("DOGE","14d", -0.0064, 0.0435,  0.0546,  0.0727),

    ("ETH","1d",  0.1864,  0.2464,  0.3114,  0.3227),
    ("ETH","2d",  0.2248,  0.3170,  0.4025,  0.4250),
    ("ETH","4d",  0.2535,  0.3384,  0.4406,  0.4517),
    ("ETH","6d",  0.2426,  0.3368,  0.3996,  0.4333),
    ("ETH","7d",  0.2724,  0.3736,  0.4186,  0.4630),
    ("ETH","14d", 0.1574,  0.4060,  0.3915,  0.5955),

    ("SOL","1d",  0.2840,  0.2764,  0.2600,  0.2593),
    ("SOL","2d",  0.2923,  0.2983,  0.2568,  0.2620),
    ("SOL","4d",  0.3340,  0.3396,  0.2912,  0.2737),
    ("SOL","6d",  0.3272,  0.3499,  0.3206,  0.3244),
    ("SOL","7d",  0.3448,  0.3775,  0.3474,  0.3638),
    ("SOL","14d", 0.3746,  0.4071,  0.4528,  0.4325),

    ("TRX","1d",  0.2400,  0.2091,  0.0315,  0.0677),
    ("TRX","2d",  0.1761,  0.1441, -0.0887, -0.0579),
    ("TRX","4d",  0.1571,  0.1643, -0.2356, -0.1576),
    ("TRX","6d",  0.1586,  0.1605, -0.3715, -0.2386),
    ("TRX","7d",  0.1443,  0.1259, -0.4563, -0.3022),
    ("TRX","14d", 0.0296, -0.2716, -0.7550, -0.8481),

    ("XRP","1d",  0.1001,  0.0980,  0.1841,  0.1536),
    ("XRP","2d",  0.0358,  0.0226,  0.1645,  0.1162),
    ("XRP","4d", -0.0082, -0.0461,  0.1255,  0.0603),
    ("XRP","6d", -0.0475, -0.0936,  0.0753, -0.0070),
    ("XRP","7d", -0.0936, -0.1472,  0.0205, -0.0741),
    ("XRP","14d",-0.2833, -0.5143, -0.3071, -0.5270),
]

cols = ["asset", "horizon", "r2_A", "r2_B", "r2_C", "r2_D"]
old_df = pd.DataFrame(old_rows, columns=cols)
new_df = pd.DataFrame(new_rows, columns=cols)

def add_deltas(df):
    df = df.copy()
    df["dB"] = df["r2_B"] - df["r2_A"]
    df["dC"] = df["r2_C"] - df["r2_A"]
    df["dD"] = df["r2_D"] - df["r2_A"]
    return df

old_df = add_deltas(old_df)
new_df = add_deltas(new_df)

common = old_df.merge(
    new_df,
    on=["asset", "horizon"],
    suffixes=("_old", "_new")
)

def print_summary(label, df):
    print(f"\n{label}")
    for cfg in ["B", "C", "D"]:
        col = f"d{cfg}"
        mean_val = df[col].mean()
        pos = (df[col] > 0).sum()
        total = len(df)
        print(f"  {cfg}: mean ΔR² = {mean_val:+.4f}   positive = {pos}/{total}")

def print_summary_merged(label, df, suffix):
    print(f"\n{label}")
    for cfg in ["B", "C", "D"]:
        col = f"d{cfg}{suffix}"
        mean_val = df[col].mean()
        pos = (df[col] > 0).sum()
        total = len(df)
        print(f"  {cfg}: mean ΔR² = {mean_val:+.4f}   positive = {pos}/{total}")

print("=" * 72)
print("SUMMARY: OLD VS NEW VOLATILITY RESULTS")
print("=" * 72)

print_summary("OLD (reported)", old_df)
print_summary("NEW (reported, includes TRX)", new_df)

print_summary_merged("OLD on common 48 tasks", common, "_old")
print_summary_merged("NEW on common 48 tasks", common, "_new")

print("\nFair comparison on common 48 tasks:")
for cfg in ["B", "C", "D"]:
    old_mean = common[f"d{cfg}_old"].mean()
    new_mean = common[f"d{cfg}_new"].mean()
    print(f"  {cfg}: {old_mean:+.4f}  ->  {new_mean:+.4f}   change = {new_mean - old_mean:+.4f}")

plt.rcParams["figure.dpi"] = 140

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

ax = axes[0, 0]
cfgs = ["B", "C", "D"]
x = np.arange(len(cfgs))
w = 0.34

old_means = [common[f"d{c}_old"].mean() for c in cfgs]
new_means = [common[f"d{c}_new"].mean() for c in cfgs]

ax.bar(x - w/2, old_means, width=w, label="Old results", alpha=0.85)
ax.bar(x + w/2, new_means, width=w, label="New results", alpha=0.85)
ax.axhline(0, color="black", lw=0.8)
ax.set_xticks(x)
ax.set_xticklabels(["+IT", "+TDA", "+IT+TDA"])
ax.set_ylabel("Mean ΔR² over baseline")
ax.set_title("Mean ΔR² by configuration\n(common 48 tasks only)")
ax.legend()
ax.grid(alpha=0.2, axis="y")
for i, v in enumerate(old_means):
    ax.text(i - w/2, v + (0.003 if v >= 0 else -0.012), f"{v:+.3f}", ha="center", fontsize=9)
for i, v in enumerate(new_means):
    ax.text(i + w/2, v + (0.003 if v >= 0 else -0.012), f"{v:+.3f}", ha="center", fontsize=9)

ax = axes[0, 1]
asset_order = ["BTC", "ETH", "SOL", "BNB", "XRP", "DOGE", "ADA", "AVAX"]
asset_cmp = common.groupby("asset")[["dD_old", "dD_new"]].mean().reindex(asset_order)

x = np.arange(len(asset_order))
ax.bar(x - w/2, asset_cmp["dD_old"], width=w, label="Old ΔR²_D", alpha=0.85)
ax.bar(x + w/2, asset_cmp["dD_new"], width=w, label="New ΔR²_D", alpha=0.85)
ax.axhline(0, color="black", lw=0.8)
ax.set_xticks(x)
ax.set_xticklabels(asset_order)
ax.set_ylabel("Mean ΔR²_D")
ax.set_title("Mean ΔR² of combined model (D) by asset")
ax.legend()
ax.grid(alpha=0.2, axis="y")

ax = axes[1, 0]
xvals = common["dD_old"].values
yvals = common["dD_new"].values

ax.scatter(xvals, yvals, alpha=0.8)
mn = min(xvals.min(), yvals.min()) - 0.02
mx = max(xvals.max(), yvals.max()) + 0.02
ax.plot([mn, mx], [mn, mx], "--", color="gray", lw=1.0)
ax.axhline(0, color="black", lw=0.7)
ax.axvline(0, color="black", lw=0.7)
ax.set_xlim(mn, mx)
ax.set_ylim(mn, mx)
ax.set_xlabel("Old ΔR²_D")
ax.set_ylabel("New ΔR²_D")
ax.set_title("Task-level comparison of combined model (D)")
ax.grid(alpha=0.2)

common["delta_gain_D"] = common["dD_new"] - common["dD_old"]
top_gain = common.nlargest(5, "delta_gain_D")
for _, row in top_gain.iterrows():
    ax.annotate(
        f"{row['asset']}-{row['horizon']}",
        (row["dD_old"], row["dD_new"]),
        fontsize=8,
        xytext=(4, 4),
        textcoords="offset points"
    )

ax = axes[1, 1]
h_order = ["1d", "2d", "4d", "6d", "7d", "14d"]

heat = (
    common.assign(delta_gain_D=common["dD_new"] - common["dD_old"])
    .pivot(index="asset", columns="horizon", values="delta_gain_D")
    .reindex(index=asset_order, columns=h_order)
)

im = ax.imshow(heat.values, aspect="auto", cmap="RdYlGn", vmin=-0.20, vmax=0.20)
ax.set_xticks(np.arange(len(h_order)))
ax.set_xticklabels(h_order)
ax.set_yticks(np.arange(len(asset_order)))
ax.set_yticklabels(asset_order)
ax.set_title("Improvement in combined model D\n(new ΔR²_D minus old ΔR²_D)")

for i in range(len(asset_order)):
    for j in range(len(h_order)):
        val = heat.iloc[i, j]
        if pd.notna(val):
            txt_color = "white" if abs(val) > 0.10 else "black"
            ax.text(j, i, f"{val:+.2f}", ha="center", va="center", fontsize=8, color=txt_color)

cbar = plt.colorbar(im, ax=ax)
cbar.set_label("Δ improvement")

plt.suptitle("Old vs new volatility results", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig("compare_old_vs_new_vol_results.png", bbox_inches="tight")
plt.savefig("compare_old_vs_new_vol_results.pdf", bbox_inches="tight")
plt.show()

print("\nSaved: compare_old_vs_new_vol_results.png / .pdf")

SUMMARY: OLD VS NEW VOLATILITY RESULTS

OLD (reported)
  B: mean ΔR² = -0.0866   positive = 14/48
  C: mean ΔR² = +0.0229   positive = 34/48
  D: mean ΔR² = -0.0436   positive = 23/48

NEW (reported, includes TRX)
  B: mean ΔR² = -0.0326   positive = 23/54
  C: mean ΔR² = -0.0038   positive = 32/54
  D: mean ΔR² = -0.0101   positive = 31/54

OLD on common 48 tasks
  B: mean ΔR² = -0.0866   positive = 14/48
  C: mean ΔR² = +0.0229   positive = 34/48
  D: mean ΔR² = -0.0436   positive = 23/48

NEW on common 48 tasks
  B: mean ΔR² = -0.0289   positive = 21/48
  C: mean ΔR² = +0.0536   positive = 32/48
  D: mean ΔR² = +0.0395   positive = 31/48

Fair comparison on common 48 tasks:
  B: -0.0866  ->  -0.0289   change = +0.0577
  C: +0.0229  ->  +0.0536   change = +0.0307
  D: -0.0436  ->  +0.0395   change = +0.0831

Saved: compare_old_vs_new_vol_results.png / .pdf


## 5. Stress-event classification (main specification)
Thresholds Q60-Q90; at W=192 the Q90 threshold is negative (ΔROC_D = -0.0073).

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.feature_selection import mutual_info_classif

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

SPLIT_DATE = "2025-07-01"
PURGE_BARS = 96

QUANTILES = [0.60, 0.70, 0.80, 0.90]
HORIZONS = {
    48:  "1d",
    96:  "2d",
    192: "4d",
}

ASSETS = [a for a in ["BTC","ETH","SOL","BNB","XRP","DOGE","ADA","AVAX"] if a in ret.columns]

VOL_WINDOW = 48
ANN = np.sqrt(365 * 48)

N_IT_FEATS  = 6
N_TDA_FEATS = 5

def _logit(C=0.1):
    return Pipeline([
        ("sc", StandardScaler()),
        ("m", LogisticRegression(
            C=C,
            max_iter=2000,
            class_weight="balanced",
            solver="lbfgs"
        ))
    ])

def safe_roc(y, s):
    try:
        return float(roc_auc_score(y, s))
    except Exception:
        return np.nan

def safe_pr(y, s):
    try:
        return float(average_precision_score(y, s))
    except Exception:
        return np.nan

def lift_at_k(y_true, score, k=0.05):
    n = len(y_true)
    top_n = max(1, int(n * k))
    top_idx = np.argsort(score)[::-1][:top_n]
    base = float(np.mean(y_true))
    return float(np.mean(y_true[top_idx])) / base if base > 1e-8 else np.nan

def fit_predict_logit(Xtr, ytr, Xte):
    try:
        m = _logit()
        m.fit(Xtr, ytr)
        return m.predict_proba(Xte)[:, 1]
    except Exception:
        return np.full(len(Xte), float(np.mean(ytr)))

print("=" * 80)
print("STRESS PREDICTION WITH THE NEW BEST FEATURES")
print("=" * 80)

if "feat_analysis" in dir() and ("X_it_aug" in feat_analysis):
    X_it_all = feat_analysis["X_it_aug"].copy()
elif "X_it_aug" in dir():
    X_it_all = X_it_aug.copy()
elif "X_it_extended" in dir():
    X_it_all = X_it_extended.copy()
elif "X_it_slim" in dir():
    X_it_all = X_it_slim.copy()
else:
    raise ValueError("No IT feature matrix found in memory (X_it_aug / X_it_extended / X_it_slim).")

if "feat_analysis" in dir() and ("X_tda_full" in feat_analysis):
    X_tda_all = feat_analysis["X_tda_full"].copy()
elif "X_tda_full" in dir():
    X_tda_all = X_tda_full.copy()
else:
    print("Building X_tda_all from tda_A and tda_B...")
    tda_frames = [tda_A.add_prefix("A_")]
    for mode in ["I", "II", "C"]:
        d = tda_B[mode]
        df_m = pd.DataFrame({
            f"B{mode}_tp_H0":  d["tp_H0"],
            f"B{mode}_ent_H0": d["ent_H0"],
            f"B{mode}_tp_H1":  d["tp_H1"],
            f"B{mode}_ent_H1": d["ent_H1"],
            f"B{mode}_L1_H0":  d["L1_H0"],
            f"B{mode}_L1_H1":  d["L1_H1"],
        }, index=pd.DatetimeIndex(d["times"]))
        tda_frames.append(df_m)

    X_tda_raw = pd.concat(tda_frames, axis=1).sort_index()
    X_tda_all = X_tda_raw.reindex(ret.index, method="ffill")

if "best_feature_sets" in dir():
    BEST_IT_GLOBAL  = list(best_feature_sets.get("BEST_IT", []))
    BEST_TDA_GLOBAL = list(best_feature_sets.get("BEST_TDA", []))
else:
    BEST_IT_GLOBAL = [
        "kl_BTC",
        "coinfo4_std",
        "kl_ADA",
        "nmi_lag1_mean",
        "coinfo3_std",
        "pairwise_mi_mean",
    ]
    BEST_TDA_GLOBAL = [
        "BI_L1_H0",
        "BC_ent_H1",
        "BII_tp_H1",
        "BII_L1_H1",
        "A_T_H0",
        "A_E_H0",
    ]

BEST_IT_GLOBAL  = [c for c in BEST_IT_GLOBAL if c in X_it_all.columns]
BEST_TDA_GLOBAL = [c for c in BEST_TDA_GLOBAL if c in X_tda_all.columns]

if len(BEST_IT_GLOBAL) == 0:
    raise ValueError("No new best IT features were found in X_it_all.")
if len(BEST_TDA_GLOBAL) == 0:
    raise ValueError("No new best TDA features were found in X_tda_all.")

print("\nGlobal IT candidate pool:")
for c in BEST_IT_GLOBAL:
    print(f"  {c}")

print("\nGlobal TDA candidate pool:")
for c in BEST_TDA_GLOBAL:
    print(f"  {c}")

print(f"\nIT pool size:  {len(BEST_IT_GLOBAL)}")
print(f"TDA pool size: {len(BEST_TDA_GLOBAL)}")

vol_now_rank = {}
for asset in ASSETS:
    vn = ret[asset].rolling(VOL_WINDOW).std() * ANN
    vn_rank = vn.expanding(min_periods=100).rank(pct=True)
    vol_now_rank[asset] = vn_rank

split_ts = pd.Timestamp(SPLIT_DATE)

print("\n" + "=" * 72)
print("STRESS PREDICTION")
print("A = vol_now | B = vol_now + best IT | C = vol_now + best TDA | D = vol_now + best IT + best TDA")
print("Feature selection is adaptive per asset on the training set")
print("=" * 72)

rows = []

for asset in ASSETS:
    r = ret[asset]
    vol_now = r.rolling(VOL_WINDOW).std() * ANN
    vn_rank = vol_now_rank[asset]

    for h, h_label in HORIZONS.items():
        vol_fwd = r.rolling(h).std().shift(-h) * ANN

        for q in QUANTILES:
            vol_q = vol_now.expanding(min_periods=100).quantile(q)
            y_raw = (vol_fwd > vol_q).astype(float).dropna()

            Xi_it = X_it_all[BEST_IT_GLOBAL].copy()
            Xi_td = X_tda_all[BEST_TDA_GLOBAL].copy()

            idx = (
                Xi_it.index
                .intersection(Xi_td.index)
                .intersection(vn_rank.index)
                .intersection(y_raw.index)
            )

            valid = (
                Xi_it.loc[idx].notna().all(1)
                & Xi_td.loc[idx].notna().all(1)
                & vn_rank.loc[idx].notna()
                & y_raw.loc[idx].notna()
            )
            idx = idx[valid]

            if len(idx) < 1000:
                continue

            tr = idx < (split_ts - pd.Timedelta(minutes=30 * PURGE_BARS))
            te = idx >= split_ts

            if tr.sum() < 480 or te.sum() < 100:
                continue

            ytr = y_raw.loc[idx[tr]]
            yte = y_raw.loc[idx[te]]

            pos_tr = float(ytr.mean())
            pos_te = float(yte.mean())

            if pos_tr < 0.03 or pos_tr > 0.97:
                continue
            if yte.sum() < 5:
                continue

            Xvol_tr_df = pd.DataFrame(
                vn_rank.loc[idx[tr]].values.reshape(-1, 1),
                index=idx[tr],
                columns=["vol_rank"]
            )
            Xvol_te_df = pd.DataFrame(
                vn_rank.loc[idx[te]].values.reshape(-1, 1),
                index=idx[te],
                columns=["vol_rank"]
            )

            Xi_tr_all = Xi_it.loc[idx[tr]]
            Xi_te_all = Xi_it.loc[idx[te]]

            Xt_tr_all = Xi_td.loc[idx[tr]]
            Xt_te_all = Xi_td.loc[idx[te]]

            mi_it = mutual_info_classif(Xi_tr_all.values, ytr.values, random_state=42)
            mi_td = mutual_info_classif(Xt_tr_all.values, ytr.values, random_state=42)

            it_cols = (
                pd.Series(mi_it, index=Xi_tr_all.columns)
                .nlargest(min(N_IT_FEATS, Xi_tr_all.shape[1]))
                .index.tolist()
            )
            td_cols = (
                pd.Series(mi_td, index=Xt_tr_all.columns)
                .nlargest(min(N_TDA_FEATS, Xt_tr_all.shape[1]))
                .index.tolist()
            )

            Xit_tr = Xi_tr_all[it_cols]
            Xit_te = Xi_te_all[it_cols]

            Xtd_tr = Xt_tr_all[td_cols]
            Xtd_te = Xt_te_all[td_cols]

            yte_np = yte.values.astype(int)

            sc_A = fit_predict_logit(Xvol_tr_df, ytr, Xvol_te_df)

            sc_B = fit_predict_logit(
                pd.concat([Xvol_tr_df, Xit_tr], axis=1),
                ytr,
                pd.concat([Xvol_te_df, Xit_te], axis=1)
            )

            sc_C = fit_predict_logit(
                pd.concat([Xvol_tr_df, Xtd_tr], axis=1),
                ytr,
                pd.concat([Xvol_te_df, Xtd_te], axis=1)
            )

            sc_D = fit_predict_logit(
                pd.concat([Xvol_tr_df, Xit_tr, Xtd_tr], axis=1),
                ytr,
                pd.concat([Xvol_te_df, Xit_te, Xtd_te], axis=1)
            )

            rows.append({
                "asset": asset,
                "horizon": h_label,
                "h": h,
                "q": q,

                "pos_tr": round(pos_tr, 3),
                "pos_te": round(pos_te, 3),

                "it_cols": ", ".join(it_cols),
                "tda_cols": ", ".join(td_cols),

                "roc_A": round(safe_roc(yte_np, sc_A), 4),
                "roc_B": round(safe_roc(yte_np, sc_B), 4),
                "roc_C": round(safe_roc(yte_np, sc_C), 4),
                "roc_D": round(safe_roc(yte_np, sc_D), 4),

                "pr_A": round(safe_pr(yte_np, sc_A), 4),
                "pr_B": round(safe_pr(yte_np, sc_B), 4),
                "pr_C": round(safe_pr(yte_np, sc_C), 4),
                "pr_D": round(safe_pr(yte_np, sc_D), 4),

                "droc_B": round(safe_roc(yte_np, sc_B) - safe_roc(yte_np, sc_A), 4),
                "droc_C": round(safe_roc(yte_np, sc_C) - safe_roc(yte_np, sc_A), 4),
                "droc_D": round(safe_roc(yte_np, sc_D) - safe_roc(yte_np, sc_A), 4),

                "dpr_B": round(safe_pr(yte_np, sc_B) - safe_pr(yte_np, sc_A), 4),
                "dpr_C": round(safe_pr(yte_np, sc_C) - safe_pr(yte_np, sc_A), 4),
                "dpr_D": round(safe_pr(yte_np, sc_D) - safe_pr(yte_np, sc_A), 4),

                "lift5_A": round(lift_at_k(yte_np, sc_A, 0.05), 3),
                "lift5_D": round(lift_at_k(yte_np, sc_D, 0.05), 3),

                "n_train": int(tr.sum()),
                "n_test": int(te.sum()),
            })

df_new_stress = pd.DataFrame(rows)
print(f"\nTotal rows: {len(df_new_stress)}")

if df_new_stress.empty:
    raise ValueError("The stress result dataframe is empty. Check intersections / NaNs / class balance.")

print("\n" + "=" * 80)
print("ROC-AUC — AGGREGATE BY THRESHOLD (MEAN OVER ASSETS AND HORIZONS)")
print("=" * 80)

print(
    f"\n  {'q':>5} {'pos%':>5}"
    f" {'ROC_A':>7} {'ROC_B':>7} {'ROC_C':>7} {'ROC_D':>7}"
    f" {'ΔROC_B':>8} {'ΔROC_C':>8} {'ΔROC_D':>8}"
    f" {'D>A':>6} {'Lift5_D':>9}"
)
print("  " + "-" * 84)

for q in QUANTILES:
    sub = df_new_stress[df_new_stress.q == q]
    if sub.empty:
        continue
    beats = (sub["droc_D"] > 0.01).sum()

    print(
        f"  Q{q:.0%}"
        f" {sub['pos_te'].mean():>4.0%}"
        f" {sub['roc_A'].mean():>7.4f}"
        f" {sub['roc_B'].mean():>7.4f}"
        f" {sub['roc_C'].mean():>7.4f}"
        f" {sub['roc_D'].mean():>7.4f}"
        f" {sub['droc_B'].mean():>+8.4f}"
        f" {sub['droc_C'].mean():>+8.4f}"
        f" {sub['droc_D'].mean():>+8.4f}"
        f" {beats:>4}/{len(sub)}"
        f" {sub['lift5_D'].mean():>9.3f}"
    )

print("\n" + "=" * 80)
print("BY ASSET — THRESHOLD Q80 | ROC-AUC + PR-AUC")
print("=" * 80)

sub80 = df_new_stress[df_new_stress.q == 0.80]

print(
    f"\n  {'asset':<6} {'pos%':>5}"
    f" {'ROC_A':>7} {'ROC_B':>7} {'ROC_C':>7} {'ROC_D':>7}"
    f" {'ΔROC_D':>8}"
    f" {'PR_A':>7} {'PR_D':>7} {'ΔPR_D':>8}  best"
)
print("  " + "-" * 88)

for asset in ASSETS:
    g = sub80[sub80.asset == asset]
    if g.empty:
        continue

    vals = {
        "A": g["roc_A"].mean(),
        "B": g["roc_B"].mean(),
        "C": g["roc_C"].mean(),
        "D": g["roc_D"].mean(),
    }
    best = max(vals, key=vals.get)

    print(
        f"  {asset:<6}"
        f" {g['pos_te'].mean():>4.0%}"
        f" {g['roc_A'].mean():>7.4f}"
        f" {g['roc_B'].mean():>7.4f}"
        f" {g['roc_C'].mean():>7.4f}"
        f" {g['roc_D'].mean():>7.4f}"
        f" {g['droc_D'].mean():>+8.4f}"
        f" {g['pr_A'].mean():>7.4f}"
        f" {g['pr_D'].mean():>7.4f}"
        f" {g['dpr_D'].mean():>+8.4f}"
        f"  -> {best}"
    )

print("\n" + "=" * 80)
print("TOP 10 BY ΔROC (CONFIGURATION D = vol_now + best IT + best TDA)")
print("=" * 80)

top10 = df_new_stress.nlargest(10, "droc_D")

print(
    f"\n  {'asset':<6} {'h':<4} {'Q':>5} {'pos%':>5}"
    f" {'ROC_A':>7} {'ROC_D':>7} {'ΔROC':>8}"
    f" {'PR_A':>7} {'PR_D':>7} {'ΔPR':>8} {'Lift5':>7}"
)
print("  " + "-" * 76)

for _, row in top10.iterrows():
    print(
        f"  {row['asset']:<6} {row['horizon']:<4} Q{row['q']:.0%}"
        f" {row['pos_te']:>4.0%}"
        f" {row['roc_A']:>7.4f}"
        f" {row['roc_D']:>7.4f}"
        f" {row['droc_D']:>+8.4f}"
        f" {row['pr_A']:>7.4f}"
        f" {row['pr_D']:>7.4f}"
        f" {row['dpr_D']:>+8.4f}"
        f" {row['lift5_D']:>7.3f}"
    )

print("\n" + "=" * 80)
print("SUMMARY")
print("=" * 80)

n = len(df_new_stress)

for cfg, col_roc, col_pr in [
    ("B: vol_now + best IT",     "droc_B", "dpr_B"),
    ("C: vol_now + best TDA",    "droc_C", "dpr_C"),
    ("D: vol_now + best IT+TDA", "droc_D", "dpr_D"),
]:
    beats_roc = (df_new_stress[col_roc] > 0.01).sum()
    beats_pr  = (df_new_stress[col_pr]  > 0.01).sum()
    print(
        f"  {cfg:<24}: ΔROC={df_new_stress[col_roc].mean():>+.4f}  "
        f"ROC↑ {beats_roc:>2}/{n}  "
        f"ΔPR={df_new_stress[col_pr].mean():>+.4f}  "
        f"PR↑ {beats_pr:>2}/{n}"
    )

print("\nBuilding plots...")

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
q_labels = [f"Q{q:.0%}" for q in QUANTILES]
colors = {
    "A": "#9E9E9E",
    "B": "#2196F3",
    "C": "#FF9800",
    "D": "#4CAF50",
}

ax = axes[0, 0]
for k in ["A", "B", "C", "D"]:
    vals = [df_new_stress[df_new_stress.q == q][f"roc_{k}"].mean() for q in QUANTILES]
    ls = "--" if k == "C" else "-"
    label = {
        "A": "vol_now",
        "B": "vol_now + best IT",
        "C": "vol_now + best TDA",
        "D": "vol_now + best IT+TDA",
    }[k]
    ax.plot(q_labels, vals, "o" + ls, color=colors[k], lw=2, label=label, ms=7)

ax.axhline(0.5, color="gray", lw=0.8, ls=":", alpha=0.5)
ax.set_ylabel("ROC-AUC")
ax.set_xlabel("Stress threshold")
ax.set_title("ROC-AUC by threshold\n(mean over assets and horizons)", fontsize=10)
ax.legend(fontsize=8)
ax.grid(alpha=0.2)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax = axes[0, 1]
x = np.arange(len(QUANTILES))
w = 0.25

for ci, k in enumerate(["B", "C", "D"]):
    vals = [df_new_stress[df_new_stress.q == q][f"droc_{k}"].mean() for q in QUANTILES]
    label = {
        "B": "+best IT",
        "C": "+best TDA",
        "D": "+best IT+TDA",
    }[k]
    ax.bar(x + (ci - 1) * w, vals, w * 0.9, label=label, color=colors[k], alpha=0.85)

ax.axhline(0, color="black", lw=0.8)
ax.set_xticks(x)
ax.set_xticklabels(q_labels)
ax.set_xlabel("Stress threshold")
ax.set_ylabel("ΔROC-AUC vs vol_now baseline")
ax.set_title("Incremental ΔROC over vol_now\n(by threshold)", fontsize=10)
ax.legend(
    fontsize=6,
    frameon=True,
    handlelength=1.0,
    handletextpad=0.4,
    borderpad=0.3,
    labelspacing=0.3
)
ax.grid(alpha=0.2, axis="y")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax = axes[1, 0]
asset_order = sub80.groupby("asset")["roc_D"].mean().sort_values(ascending=False).index.tolist()
x3 = np.arange(len(asset_order))
w3 = 0.2

for ci, k in enumerate(["A", "B", "C", "D"]):
    vals = [sub80[sub80.asset == a][f"roc_{k}"].mean() for a in asset_order]
    label = {
        "A": "vol_now",
        "B": "vol_now + IT",
        "C": "vol_now + TDA",
        "D": "vol_now + IT + TDA",
    }[k]
    ax.bar(x3 + (ci - 1.5) * w3, vals, w3 * 0.9, label=label, color=colors[k], alpha=0.85)

ax.axhline(0.5, color="gray", lw=0.8, ls=":", alpha=0.5)
ax.set_xticks(x3)
ax.set_xticklabels(asset_order, fontsize=9)
ax.set_ylabel("ROC-AUC")
ax.set_title("ROC-AUC by asset (Q80)\n(mean over horizons)", fontsize=10)
ax.legend(
    fontsize=8,
    loc="upper right",
    frameon=True,
    handlelength=0.7,
    handleheight=0.5,
    handletextpad=0.25,
    borderpad=0.2,
    labelspacing=0.15,
    borderaxespad=0.2,
    columnspacing=0.4
)
ax.grid(alpha=0.2, axis="y")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax = axes[1, 1]
pivot = df_new_stress.pivot_table(index="asset", columns="q", values="roc_D", aggfunc="mean")
pivot = pivot.reindex(ASSETS)[QUANTILES]

im = ax.imshow(pivot.values, cmap="RdYlGn", vmin=0.50, vmax=0.85, aspect="auto")
ax.set_xticks(range(len(QUANTILES)))
ax.set_xticklabels(q_labels)
ax.set_yticks(range(len(ASSETS)))
ax.set_yticklabels(ASSETS)

for i in range(len(ASSETS)):
    for j in range(len(QUANTILES)):
        val = pivot.values[i, j]
        if np.isfinite(val):
            text_color = "white" if val > 0.72 else "black"
            ax.text(j, i, f"{val:.3f}", ha="center", va="center", fontsize=8, color=text_color)

plt.colorbar(im, ax=ax, label="ROC-AUC (vol_now + best IT + best TDA)")
ax.set_title("ROC-AUC heatmap for configuration D\n(by asset × threshold)", fontsize=10)

plt.suptitle(
    "Stress prediction with the new best features\n"
    f"A=vol_now | B=vol_now+best IT | C=vol_now+best TDA | D=vol_now+best IT+TDA\n"
    f"Holdout {SPLIT_DATE} -> | Adaptive per-asset selection | Horizons: 1d, 2d, 4d",
    fontsize=11,
    y=1.01
)

plt.tight_layout()
plt.savefig("fig_stress_best_features.png", bbox_inches="tight", dpi=150)
plt.savefig("fig_stress_best_features.pdf", bbox_inches="tight", dpi=150)
plt.show()

print("Saved: fig_stress_best_features.png / .pdf")

stress_best_features = df_new_stress.copy()
stress_best_feature_sets = {
    "BEST_IT_GLOBAL": BEST_IT_GLOBAL,
    "BEST_TDA_GLOBAL": BEST_TDA_GLOBAL,
}

print(f"\nstress_best_features saved in memory ({len(df_new_stress)} rows)")
print("stress_best_feature_sets saved in memory")

STRESS PREDICTION WITH THE NEW BEST FEATURES

Global IT candidate pool:
  kl_BTC
  coinfo4_std
  kl_ADA
  nmi_lag1_mean
  coinfo3_std
  pairwise_mi_mean

Global TDA candidate pool:
  BI_L1_H0
  BC_ent_H1
  BII_tp_H1
  BII_L1_H1
  A_T_H0
  A_E_H0

IT pool size:  6
TDA pool size: 6

STRESS PREDICTION
A = vol_now | B = vol_now + best IT | C = vol_now + best TDA | D = vol_now + best IT + best TDA
Feature selection is adaptive per asset on the training set

Total rows: 80

ROC-AUC — AGGREGATE BY THRESHOLD (MEAN OVER ASSETS AND HORIZONS)

      q  pos%   ROC_A   ROC_B   ROC_C   ROC_D   ΔROC_B   ΔROC_C   ΔROC_D    D>A   Lift5_D
  ------------------------------------------------------------------------------------
  Q60%  46%  0.6840  0.6863  0.7103  0.7117  +0.0023  +0.0263  +0.0278   15/23     1.598
  Q70%  32%  0.6832  0.6848  0.7002  0.7030  +0.0016  +0.0171  +0.0199   17/22     1.846
  Q80%  17%  0.6608  0.6917  0.6722  0.7011  +0.0309  +0.0115  +0.0403   14/21     2.760
  Q90%   5%  0.62

## 6. IT window length: features at W = 96, then sensitivity over W ∈ {96, 144, 192, 384}
W = 96 is selected as the best window on the same holdout. The window-sensitivity builder (cell below) does not create `nmi_lag1_mean` and `entropy_index_1y`; this is the likely reason its W = 192 row (B: -0.0178, D: +0.0459) differs from section 3 (B: -0.0289, D: +0.0395).

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from itertools import combinations
from sklearn.metrics import mutual_info_score
import gudhi
from gudhi.representations import Landscape

FEATURE_WINDOW = 96
FEATURE_STEP = 4
TDA_STEP = 48
BINS = 10

ASSETS_FEATURE = [a for a in ["BTC","ETH","SOL","BNB","XRP","DOGE","ADA","AVAX","TRX"] if a in ret.columns]

print("=" * 80)
print(f"BUILDING W={FEATURE_WINDOW} FEATURES")
print("=" * 80)

def _rank_bins(x, bins=BINS):
    x = np.asarray(x, dtype=float)
    ranks = pd.Series(x).rank(method="average").values
    scaled = np.floor((ranks - 1) / max(1, len(x)) * bins).astype(int)
    return np.clip(scaled, 0, bins - 1)

def mi_hist_rank(x, y, bins=BINS):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    x, y = x[mask], y[mask]
    if len(x) < max(20, bins):
        return 0.0
    bx = _rank_bins(x, bins)
    by = _rank_bins(y, bins)
    c = np.histogram2d(bx, by, bins=bins, range=[[0, bins], [0, bins]])[0]
    return float(mutual_info_score(None, None, contingency=c))

def kl_hist_rank(x_now, x_prev, bins=BINS):
    bx = _rank_bins(x_now, bins)
    bp = _rank_bins(x_prev, bins)
    p = np.bincount(bx, minlength=bins).astype(float) + 1e-8
    q = np.bincount(bp, minlength=bins).astype(float) + 1e-8
    p /= p.sum()
    q /= q.sum()
    return float(np.sum(p * np.log(p / q)))

def build_it_full_for_window(ret_df, assets, window=96, step=4, bins=10):
    avail = [a for a in assets if a in ret_df.columns]
    R = ret_df[avail].replace([np.inf, -np.inf], np.nan).fillna(0.0).values
    idx = ret_df.index
    n = len(R)

    pairs = list(combinations(range(len(avail)), 2))
    triples = list(combinations(range(len(avail)), 3))
    quads = list(combinations(range(len(avail)), 4))

    rows = []
    times = []

    for s in range(0, n - window + 1, step):
        e = s + window
        wnd = R[s:e]

        mi_vals = []
        cov_vals = []

        for i, j in pairs:
            mi_vals.append(mi_hist_rank(wnd[:, i], wnd[:, j], bins=bins))
            cov_vals.append(float(np.cov(wnd[:, i], wnd[:, j])[0, 1]))

        coinfo3_vals = []
        for i, j, k in triples:
            mij = mi_hist_rank(wnd[:, i], wnd[:, j], bins=bins)
            mik = mi_hist_rank(wnd[:, i], wnd[:, k], bins=bins)
            mjk = mi_hist_rank(wnd[:, j], wnd[:, k], bins=bins)
            mijk_proxy = mi_hist_rank(wnd[:, i], wnd[:, j] + wnd[:, k], bins=bins)
            coinfo3_vals.append(mij + mik + mjk - mijk_proxy)

        coinfo4_vals = []
        for i, j, k, l in quads[:50]:
            a = mi_hist_rank(wnd[:, i], wnd[:, j], bins=bins)
            b = mi_hist_rank(wnd[:, k], wnd[:, l], bins=bins)
            coinfo4_vals.append(a - b)

        row = {
            "pairwise_mi_mean": float(np.mean(mi_vals)),
            "pairwise_mi_std":  float(np.std(mi_vals)),
            "pairwise_mi_min":  float(np.min(mi_vals)),
            "pairwise_mi_max":  float(np.max(mi_vals)),

            "pairwise_cov_mean": float(np.mean(cov_vals)),
            "pairwise_cov_std":  float(np.std(cov_vals)),
            "pairwise_cov_min":  float(np.min(cov_vals)),
            "pairwise_cov_max":  float(np.max(cov_vals)),

            "coinfo3_mean": float(np.mean(coinfo3_vals)) if coinfo3_vals else 0.0,
            "coinfo3_std":  float(np.std(coinfo3_vals)) if coinfo3_vals else 0.0,
            "coinfo3_min":  float(np.min(coinfo3_vals)) if coinfo3_vals else 0.0,
            "coinfo3_max":  float(np.max(coinfo3_vals)) if coinfo3_vals else 0.0,

            "coinfo4_mean": float(np.mean(coinfo4_vals)) if coinfo4_vals else 0.0,
            "coinfo4_std":  float(np.std(coinfo4_vals)) if coinfo4_vals else 0.0,
            "coinfo4_min":  float(np.min(coinfo4_vals)) if coinfo4_vals else 0.0,
            "coinfo4_max":  float(np.max(coinfo4_vals)) if coinfo4_vals else 0.0,
        }

        for ai, asset in enumerate(avail):
            if s >= window:
                row[f"kl_{asset}"] = kl_hist_rank(R[s:e, ai], R[s-window:s, ai], bins=bins)
            else:
                row[f"kl_{asset}"] = 0.0

        rows.append(row)
        times.append(idx[e - 1])

    return pd.DataFrame(rows, index=pd.DatetimeIndex(times)).sort_index()

X_it_full_W96 = build_it_full_for_window(
    ret_df=ret,
    assets=ASSETS_FEATURE,
    window=FEATURE_WINDOW,
    step=FEATURE_STEP,
    bins=BINS,
)

X_it_full_W96 = X_it_full_W96.reindex(ret.index).ffill()

print(f"X_it_full_W96: {X_it_full_W96.shape}")

def diag_for_dim(diag, dim):
    return [(float(b), float(d)) for dd, (b, d) in diag if dd == dim]

def diagram_statistics(diag_bd, max_filtration):
    lifetimes = []
    for b, d in diag_bd:
        lt = (d - b) if np.isfinite(d) else (max_filtration - b)
        if lt > 0:
            lifetimes.append(lt)

    if not lifetimes:
        return 0.0, 0.0

    lts = np.asarray(lifetimes, dtype=float)
    T = float(lts.sum())
    p = lts / T
    E = float(-(p * np.log(p)).sum())
    return T, E

def landscape_norms(diag_bd, max_filtration, resolution=200):
    if len(diag_bd) == 0:
        return 0.0, 0.0

    pts = [[b, d if np.isfinite(d) else max_filtration] for b, d in diag_bd]
    arr = np.asarray(pts, dtype=float)
    arr = arr[np.isfinite(arr).all(axis=1)]
    arr = arr[arr[:, 1] > arr[:, 0]]

    if len(arr) == 0:
        return 0.0, 0.0

    try:
        L = Landscape(num_landscapes=1, resolution=resolution)
        vec = L.fit_transform([arr])[0]
        return float(np.sum(np.abs(vec))), float(np.sqrt(np.sum(vec ** 2)))
    except Exception:
        return 0.0, 0.0

def H_gaussian(S):
    S = np.asarray(S)
    if S.ndim == 0:
        return 0.5 * np.log(2 * np.pi * np.e * float(S)) if float(S) > 0 else 0.0

    n = S.shape[0]
    det = np.linalg.det(S)
    if det <= 0 or not np.isfinite(det):
        return 0.0
    return float(0.5 * np.log((2 * np.pi * np.e) ** n * det))

def I(simplex, A=None):
    d = len(simplex)
    H_marginal_sum = d * H_gaussian(A[0, 0])
    H_joint = H_gaussian(A[np.ix_(simplex, simplex)])
    return H_marginal_sum - H_joint

def II(simplex, A=None):
    d = len(simplex)
    H_joint = H_gaussian(A[np.ix_(simplex, simplex)])
    H_residual_sum = 0.0
    for idx in combinations(simplex, d - 1):
        H_residual_sum += H_gaussian(A[np.ix_(list(idx), list(idx))])
    return -(d - 1) * H_joint + H_residual_sum

def C(simplex, A=None):
    d = len(simplex)
    coeffs = []
    vals = []
    for l in range(1, d + 1):
        sign = 0 - (-1) ** l
        for idx in combinations(simplex, l):
            idx_list = list(idx)
            coeffs.append(sign)
            if len(idx_list) == 1:
                vals.append(H_gaussian(A[idx_list[0], idx_list[0]]))
            else:
                vals.append(H_gaussian(A[np.ix_(idx_list, idx_list)]))
    return float(np.asarray(coeffs) @ np.asarray(vals))

def build_mi_complex_gudhi(X_window, bins=BINS):
    dim = X_window.shape[1]
    MI = np.zeros((dim, dim))

    for i, j in combinations(range(dim), 2):
        mi = mi_hist_rank(X_window[:, i], X_window[:, j], bins=bins)
        MI[i, j] = MI[j, i] = mi

    triu = MI[np.triu_indices_from(MI, 1)]
    MI_max = float(np.nanmax(triu)) if len(triu) else 1.0

    st = gudhi.SimplexTree()

    for i in range(dim):
        st.insert([i], filtration=0.0)

    edge_w = np.zeros((dim, dim))
    for i, j in combinations(range(dim), 2):
        w = MI_max - MI[i, j]
        edge_w[i, j] = edge_w[j, i] = w
        st.insert([i, j], filtration=float(w))

    for i, j, k in combinations(range(dim), 3):
        w = max(edge_w[i, j], edge_w[i, k], edge_w[j, k])
        st.insert([i, j, k], filtration=float(w))

    st.make_filtration_non_decreasing()
    return st, MI, MI_max

def build_gaussian_info_complex(X_window, triple_mode="I", eps=1e-8):
    dim = X_window.shape[1]
    A = np.cov(X_window, rowvar=False) + eps * np.eye(dim)
    triple_fn = {"I": I, "II": II, "C": C}[triple_mode]

    st = gudhi.SimplexTree()

    for i in range(dim):
        st.insert([i], filtration=0.0)

    for i, j in combinations(range(dim), 2):
        st.insert([i, j], filtration=float(I([i, j], A=A)))

    for i, j, k in combinations(range(dim), 3):
        st.insert([i, j, k], filtration=float(triple_fn([i, j, k], A=A)))

    st.make_filtration_non_decreasing()
    return st, A

def build_tda_A_B_for_window(ret_df, assets, window=96, step=48, bins=10):
    avail = [a for a in assets if a in ret_df.columns]
    R = ret_df[avail].replace([np.inf, -np.inf], np.nan).fillna(0.0).values
    idx = ret_df.index
    n = len(R)

    starts = list(range(0, n - window, step))
    times = [idx[t + window] for t in starts]

    print(f"TDA windows: {len(starts)} | {times[0]} -> {times[-1]}")

    rows_A = []

    for t in starts:
        wnd = R[t:t + window]

        try:
            st, MI, MI_max = build_mi_complex_gudhi(wnd, bins=bins)
            diag = st.persistence()

            d0 = diag_for_dim(diag, 0)
            d1 = diag_for_dim(diag, 1)

            T0, E0 = diagram_statistics(d0, MI_max)
            T1, E1 = diagram_statistics(d1, MI_max)

            L1_0, L2_0 = landscape_norms(d0, MI_max)
            L1_1, L2_1 = landscape_norms(d1, MI_max)

        except Exception:
            T0 = E0 = T1 = E1 = L1_0 = L2_0 = L1_1 = L2_1 = 0.0

        rows_A.append({
            "T_H0": T0,
            "E_H0": E0,
            "T_H1": T1,
            "E_H1": E1,
            "L1_H0": L1_0,
            "L2_H0": L2_0,
            "L1_H1": L1_1,
            "L2_H1": L2_1,
        })

    tda_A_w = pd.DataFrame(rows_A, index=pd.DatetimeIndex(times))

    global_max = 0.0

    for t in starts:
        wnd = R[t:t + window]
        try:
            st, A = build_gaussian_info_complex(wnd, triple_mode="I")
            diag = st.persistence()
            finite_deaths = [d for _, (_, d) in diag if np.isfinite(d)]
            if finite_deaths:
                global_max = max(global_max, max(finite_deaths))
        except Exception:
            pass

    global_max = float(global_max) if global_max > 0 else 1.0

    tda_B_w = {}

    for mode in ["I", "II", "C"]:
        rows = []

        for t in starts:
            wnd = R[t:t + window]

            try:
                st, A = build_gaussian_info_complex(wnd, triple_mode=mode)
                diag = st.persistence()

                d0 = diag_for_dim(diag, 0)
                d1 = diag_for_dim(diag, 1)

                T0, E0 = diagram_statistics(d0, global_max)
                T1, E1 = diagram_statistics(d1, global_max)

                L1_0, L2_0 = landscape_norms(d0, global_max)
                L1_1, L2_1 = landscape_norms(d1, global_max)

            except Exception:
                T0 = E0 = T1 = E1 = L1_0 = L2_0 = L1_1 = L2_1 = 0.0

            rows.append({
                "tp_H0": T0,
                "ent_H0": E0,
                "tp_H1": T1,
                "ent_H1": E1,
                "L1_H0": L1_0,
                "L2_H0": L2_0,
                "L1_H1": L1_1,
                "L2_H1": L2_1,
            })

        df_mode = pd.DataFrame(rows, index=pd.DatetimeIndex(times))

        tda_B_w[mode] = {
            "times": list(pd.DatetimeIndex(times)),
            "tp_H0": df_mode["tp_H0"].values,
            "ent_H0": df_mode["ent_H0"].values,
            "tp_H1": df_mode["tp_H1"].values,
            "ent_H1": df_mode["ent_H1"].values,
            "L1_H0": df_mode["L1_H0"].values,
            "L2_H0": df_mode["L2_H0"].values,
            "L1_H1": df_mode["L1_H1"].values,
            "L2_H1": df_mode["L2_H1"].values,
        }

    return tda_A_w, tda_B_w

tda_A_W96, tda_B_W96 = build_tda_A_B_for_window(
    ret_df=ret,
    assets=ASSETS_FEATURE,
    window=FEATURE_WINDOW,
    step=TDA_STEP,
    bins=BINS,
)

print(f"tda_A_W96: {tda_A_W96.shape}")
print("tda_B_W96 modes:", list(tda_B_W96.keys()))

BUILDING W=96 FEATURES
X_it_full_W96: (46743, 25)
TDA windows: 972 | 2023-01-03 00:00:00 -> 2025-08-31 05:00:00
tda_A_W96: (972, 8)
tda_B_W96 modes: ['I', 'II', 'C']


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import mutual_info_score
from sklearn.feature_selection import mutual_info_regression

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

FEATURE_WINDOW = 96

SPLIT_DATE = "2025-01-01"
PURGE_BARS = 96

HORIZONS = {
    96:  "2d",
    192: "4d",
}

ASSETS = ["BTC", "ETH", "DOGE", "XRP", "SOL", "BNB"]

VOL_WINDOW = 48
ANN = np.sqrt(365 * 48)

RNG = np.random.default_rng(42)
N_PERM = 10

IT_WINDOW = FEATURE_WINDOW
IT_STEP = 4

INDEX_WINDOW_BARS = 365 * 48
INDEX_LAG_BARS = 48
INDEX_FEATURE_STEP = 48
ENTROPY_BINS = 32

MI_ESTIMATORS = {
    "hist_rank": {"kind": "hist_rank", "bins": 10},
    "gaussian": {"kind": "gaussian"},
    "knn": {"kind": "knn", "n_neighbors": 5, "random_state": 42},
}

EXTRA_MI_FEATURES = ["gaussian", "knn"]
INDEX_NMI_ESTIMATOR = "hist_rank"
INDEX_FEATURE_TARGETS = {"index": None}
INDEX_WEIGHTS = None
EXTRA_TDA_FROM_MI = []

def _ridge():
    return Pipeline([
        ("sc", StandardScaler()),
        ("m", RidgeCV(alphas=[0.01, 0.1, 1, 10, 100]))
    ])

def oos_r2(y_true, y_pred, naive_val):
    mse_m = np.mean((y_true - y_pred) ** 2)
    mse_n = np.mean((y_true - naive_val) ** 2)
    return float(1.0 - mse_m / mse_n) if mse_n > 1e-12 else np.nan

def _drop_finite_pairs(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    return x[mask], y[mask]

def _rank_to_bins(x, bins=10):
    x = np.asarray(x, dtype=float)
    ranks = pd.Series(x).rank(method="average").values
    scaled = np.floor((ranks - 1) / max(1, len(x)) * bins).astype(int)
    return np.clip(scaled, 0, bins - 1)

def mi_hist_rank(x, y, bins=10):
    x, y = _drop_finite_pairs(x, y)
    if len(x) < max(20, bins):
        return np.nan
    bx = _rank_to_bins(x, bins=bins)
    by = _rank_to_bins(y, bins=bins)
    c = np.histogram2d(bx, by, bins=bins, range=[[0, bins], [0, bins]])[0]
    return float(mutual_info_score(None, None, contingency=c))

def mi_gaussian(x, y):
    x, y = _drop_finite_pairs(x, y)
    if len(x) < 10:
        return np.nan
    r = np.corrcoef(x, y)[0, 1]
    if not np.isfinite(r):
        return np.nan
    r = np.clip(r, -0.999999, 0.999999)
    return float(-0.5 * np.log(1.0 - r * r))

def mi_knn_sym(x, y, n_neighbors=5, random_state=42):
    x, y = _drop_finite_pairs(x, y)
    if len(x) < max(20, n_neighbors + 2):
        return np.nan

    x2 = x.reshape(-1, 1)
    y2 = y.reshape(-1, 1)

    mi_xy = mutual_info_regression(
        x2, y,
        discrete_features=False,
        n_neighbors=n_neighbors,
        random_state=random_state
    )[0]

    mi_yx = mutual_info_regression(
        y2, x,
        discrete_features=False,
        n_neighbors=n_neighbors,
        random_state=random_state
    )[0]

    return float(0.5 * (mi_xy + mi_yx))

def estimate_mi(x, y, estimator_name="hist_rank"):
    cfg = MI_ESTIMATORS[estimator_name]
    kind = cfg["kind"]

    if kind == "hist_rank":
        return mi_hist_rank(x, y, bins=cfg.get("bins", 10))

    if kind == "gaussian":
        return mi_gaussian(x, y)

    if kind == "knn":
        return mi_knn_sym(
            x, y,
            n_neighbors=cfg.get("n_neighbors", 5),
            random_state=cfg.get("random_state", 42)
        )

    raise ValueError(f"Unknown MI estimator: {estimator_name}")

def nmi_from_mi(mi):
    if not np.isfinite(mi):
        return np.nan
    return float(1.0 - np.exp(-2.0 * mi))

def build_weighted_index_returns(ret_df, assets, weights=None):
    avail = [a for a in assets if a in ret_df.columns]

    if weights is None:
        w = pd.Series(1.0 / len(avail), index=avail)
    else:
        w = pd.Series(weights, dtype=float).reindex(avail).fillna(0.0)
        w = w / w.sum()

    idx = ret_df[avail].mul(w, axis=1).sum(axis=1)
    idx.name = "weighted_index_ret"
    return idx

def _rolling_hist_entropy_raw(series, window, step, bins=32):
    x = series.astype(float).values
    n = len(x)
    res = np.full(n, np.nan)

    for s in np.arange(0, n - window + 1, step):
        e = s + window
        xw = x[s:e]
        xw = xw[np.isfinite(xw)]

        if len(xw) < 100:
            continue

        counts, _ = np.histogram(xw, bins=bins)
        p = counts.astype(float)

        if p.sum() <= 0:
            continue

        p /= p.sum()
        p = p[p > 0]
        res[e - 1] = -np.sum(p * np.log(p))

    return pd.Series(res, index=series.index).ffill()

def _rolling_nmi_lag(series, window, lag_bars, step, estimator_name):
    x = series.astype(float).values
    n = len(x)
    res = np.full(n, np.nan)

    for s in np.arange(0, n - window + 1, step):
        e = s + window
        xw = x[s:e]

        if len(xw) <= lag_bars + 5:
            continue

        x_now = xw[lag_bars:]
        x_lag = xw[:-lag_bars]

        mi = estimate_mi(x_now, x_lag, estimator_name=estimator_name)
        res[e - 1] = nmi_from_mi(mi)

    return pd.Series(res, index=series.index).ffill()

def build_index_feature_block(ret_df, assets, targets_dict, weights=None):
    frames = []
    weighted_index = build_weighted_index_returns(ret_df, assets, weights=weights)

    for target_name, target_spec in targets_dict.items():
        if target_spec is None:
            s = weighted_index.copy()
            base_name = target_name
        else:
            if target_spec not in ret_df.columns:
                continue
            s = ret_df[target_spec].copy()
            base_name = target_name

        ent = _rolling_hist_entropy_raw(
            s,
            window=INDEX_WINDOW_BARS,
            step=INDEX_FEATURE_STEP,
            bins=ENTROPY_BINS
        )
        ent.name = f"entropy_{base_name}_1y"

        nmi = _rolling_nmi_lag(
            s,
            window=INDEX_WINDOW_BARS,
            lag_bars=INDEX_LAG_BARS,
            step=INDEX_FEATURE_STEP,
            estimator_name=INDEX_NMI_ESTIMATOR
        )
        nmi.name = f"nmi_{base_name}_1d_{INDEX_NMI_ESTIMATOR}"

        frames.append(ent.to_frame())
        frames.append(nmi.to_frame())

    return pd.concat(frames, axis=1).reindex(ret_df.index).ffill() if frames else pd.DataFrame(index=ret_df.index)

def build_pairwise_mi_aggregates(ret_df, assets, estimator_name, window=96, step=4, prefix=None):
    avail = [a for a in assets if a in ret_df.columns]
    idx = ret_df.index
    n = len(idx)

    if prefix is None:
        prefix = estimator_name

    out = pd.DataFrame(index=idx, columns=[
        f"{prefix}_pairwise_mi_mean",
        f"{prefix}_pairwise_mi_std",
        f"{prefix}_pairwise_mi_max",
        f"{prefix}_pairwise_mi_min",
    ], dtype=float)

    pairs = [(avail[i], avail[j]) for i in range(len(avail)) for j in range(i + 1, len(avail))]

    for s in np.arange(0, n - window + 1, step):
        e = s + window
        sub = ret_df.iloc[s:e]
        vals = []

        for a, b in pairs:
            mi = estimate_mi(sub[a].values, sub[b].values, estimator_name=estimator_name)
            if np.isfinite(mi):
                vals.append(mi)

        if vals:
            vals = np.asarray(vals, dtype=float)
            out.iloc[e - 1] = [vals.mean(), vals.std(ddof=0), vals.max(), vals.min()]

    return out.ffill()

print("=" * 80)
print("FEATURE ANALYSIS W=96")
print("=" * 80)

tda_frames = [tda_A_W96.add_prefix("A_")]

for mode in ["I", "II", "C"]:
    d = tda_B_W96[mode]
    idx_tda = pd.DatetimeIndex(d["times"])

    df_m = pd.DataFrame({
        f"B{mode}_tp_H0":  d["tp_H0"],
        f"B{mode}_ent_H0": d["ent_H0"],
        f"B{mode}_tp_H1":  d["tp_H1"],
        f"B{mode}_ent_H1": d["ent_H1"],
        f"B{mode}_L1_H0":  d["L1_H0"],
        f"B{mode}_L1_H1":  d["L1_H1"],
    }, index=idx_tda)

    tda_frames.append(df_m)

X_tda_raw_W96 = pd.concat(tda_frames, axis=1).sort_index()
X_tda_full_W96 = X_tda_raw_W96.reindex(ret.index, method="ffill")

X_it_base_W96 = X_it_full_W96.copy()

PREDEFINED_IT_SLIM = [
    "coinfo3_std", "kl_BNB", "coinfo4_std", "pairwise_mi_mean",
    "coinfo4_mean", "pairwise_mi_std", "coinfo3_mean",
    "coinfo4_max", "kl_AVAX", "pairwise_mi_min"
]

IT_SLIM_W96 = [c for c in PREDEFINED_IT_SLIM if c in X_it_base_W96.columns]
IT_OTHER_BASE_W96 = [c for c in X_it_base_W96.columns if c not in IT_SLIM_W96]

extra_it_frames = []
extra_it_cols_W96 = []

for est_name in EXTRA_MI_FEATURES:
    print(f"Building extra standalone MI features W96: {est_name}")
    df_extra = build_pairwise_mi_aggregates(
        ret_df=ret,
        assets=ASSETS,
        estimator_name=est_name,
        window=IT_WINDOW,
        step=IT_STEP,
        prefix=est_name,
    )
    extra_it_frames.append(df_extra)
    extra_it_cols_W96.extend(list(df_extra.columns))

print("Building index / entropy / NMI features...")
df_index_extra = build_index_feature_block(
    ret_df=ret,
    assets=ASSETS,
    targets_dict=INDEX_FEATURE_TARGETS,
    weights=INDEX_WEIGHTS,
)

if len(df_index_extra.columns) > 0:
    extra_it_frames.append(df_index_extra)
    extra_it_cols_W96.extend(list(df_index_extra.columns))

X_it_aug_W96 = pd.concat([X_it_base_W96] + extra_it_frames, axis=1)
X_it_aug_W96 = X_it_aug_W96.reindex(ret.index).ffill()
X_it_aug_W96 = X_it_aug_W96.loc[:, ~X_it_aug_W96.columns.duplicated()]

X_tda_full_W96 = X_tda_full_W96.loc[:, ~X_tda_full_W96.columns.duplicated()]

IT_ALL_W96 = list(X_it_aug_W96.columns)
TDA_COLS_W96 = list(X_tda_full_W96.columns)

print(f"Base IT W96:      {len(X_it_base_W96.columns)}")
print(f"IT slim W96:      {len(IT_SLIM_W96)}")
print(f"Extra IT W96:     {len(extra_it_cols_W96)}")
print(f"Total IT W96:     {len(IT_ALL_W96)}")
print(f"Total TDA W96:    {len(TDA_COLS_W96)}")

split_ts = pd.Timestamp(SPLIT_DATE)
vol_btc = ret["BTC"].rolling(VOL_WINDOW).std() * ANN
vol_test = vol_btc[vol_btc.index >= split_ts]

it_corrs_W96 = {}
for col in IT_ALL_W96:
    s = X_it_aug_W96[col].reindex(vol_test.index, method="ffill")
    it_corrs_W96[col] = float(s.corr(vol_test))

tda_corrs_W96 = {}
for col in TDA_COLS_W96:
    s = X_tda_full_W96[col].reindex(vol_test.index, method="ffill")
    tda_corrs_W96[col] = float(s.corr(vol_test))

print("\n" + "=" * 65)
print("PERMUTATION IMPORTANCE W=96 ON TRAIN")
print("=" * 65)

perm_results_W96 = {}

for pilot_asset in ["BTC", "ETH"]:
    r = ret[pilot_asset]
    vol_fwd = r.rolling(192).std().shift(-192) * ANN
    y_logvol = np.log(vol_fwd.clip(lower=1e-8)).dropna()

    idx = (
        X_vol.index
        .intersection(X_it_aug_W96.index)
        .intersection(X_tda_full_W96.index)
        .intersection(y_logvol.index)
    )

    valid = (
        X_vol.loc[idx].notna().all(1)
        & X_it_aug_W96.loc[idx].notna().all(1)
        & X_tda_full_W96.loc[idx].notna().all(1)
        & y_logvol.loc[idx].notna()
    )

    idx = idx[valid]
    tr = idx < (split_ts - pd.Timedelta(minutes=30 * PURGE_BARS))
    idx_tr = idx[tr]

    Xv_tr = X_vol.loc[idx_tr]
    ytr = y_logvol.loc[idx_tr]

    ck = Xv_tr.corrwith(ytr).abs().dropna()
    top3 = ck.nlargest(3).index.tolist()

    mdl_b = _ridge()
    mdl_b.fit(Xv_tr[top3], ytr)

    res_tr = ytr.values - mdl_b.predict(Xv_tr[top3])

    Xi_all_tr = pd.concat([
        X_it_aug_W96.loc[idx_tr],
        X_tda_full_W96.loc[idx_tr],
    ], axis=1)

    Xi_all_tr = Xi_all_tr.loc[:, ~Xi_all_tr.columns.duplicated()]

    Xi_arr = Xi_all_tr.values
    feat_names = list(Xi_all_tr.columns)

    kf = KFold(n_splits=5, shuffle=False)

    base_cv = float(np.mean(
        cross_val_score(_ridge(), Xi_arr, res_tr, cv=kf, scoring="r2")
    ))

    perm_scores_asset = {}

    for fi, feat in enumerate(feat_names):
        deltas = []

        for _ in range(N_PERM):
            Xp = Xi_arr.copy()
            Xp[:, fi] = RNG.permutation(Xp[:, fi])

            sc = float(np.mean(
                cross_val_score(_ridge(), Xp, res_tr, cv=kf, scoring="r2")
            ))

            deltas.append(sc - base_cv)

        perm_scores_asset[feat] = float(np.mean(deltas))

    perm_results_W96[pilot_asset] = perm_scores_asset
    print(f"{pilot_asset} baseline CV R² = {base_cv:.4f}")

all_feats_W96 = list(perm_results_W96["BTC"].keys())

avg_perm_W96 = {
    f: np.mean([perm_results_W96[a].get(f, 0.0) for a in ["BTC", "ETH"]])
    for f in all_feats_W96
}

sorted_perm_W96 = sorted(avg_perm_W96.items(), key=lambda x: x[1])

def categorize_W96(feat):
    if feat in IT_SLIM_W96:
        return "IT_slim"
    if feat in extra_it_cols_W96:
        return "IT_extra"
    if feat in IT_OTHER_BASE_W96:
        return "IT_base_other"
    if feat.startswith("A_"):
        return "TDA_A"
    if feat.startswith("BI_"):
        return "TDA_BI"
    if feat.startswith("BII_"):
        return "TDA_BII"
    if feat.startswith("BC_"):
        return "TDA_BC"
    return "other"

print("\nTop-15 useful W96 features:")
for feat, delta in sorted_perm_W96[:15]:
    print(f"  {feat:<36} {delta:+.4f} [{categorize_W96(feat)}]")

print("\nTop-10 harmful W96 features:")
for feat, delta in sorted(avg_perm_W96.items(), key=lambda x: -x[1])[:10]:
    print(f"  {feat:<36} {delta:+.4f} [{categorize_W96(feat)}]")

best_tda_W96 = [f for f in TDA_COLS_W96 if avg_perm_W96.get(f, 0) < -0.002]
best_it_W96 = [f for f in IT_ALL_W96 if avg_perm_W96.get(f, 0) < -0.002]

print(f"\nBest TDA W96: {best_tda_W96[:8]}")
print(f"Best IT W96:  {best_it_W96[:12]}")

CONFIGS_W96 = {
    "slim_IT": IT_SLIM_W96,
    "best_IT": [f for f, d in sorted_perm_W96[:20] if f in IT_ALL_W96],
    "slim+TDA": IT_SLIM_W96 + best_tda_W96[:6],
    "best+TDA": [f for f, d in sorted_perm_W96[:16] if f in IT_ALL_W96 or f in TDA_COLS_W96],
}

CONFIGS_W96 = {k: list(dict.fromkeys(v)) for k, v in CONFIGS_W96.items()}

config_rows = []

for asset in ASSETS:
    if asset not in ret.columns:
        continue

    r = ret[asset]

    for h, h_label in HORIZONS.items():
        vol_fwd = r.rolling(h).std().shift(-h) * ANN
        y_raw = np.log(vol_fwd.clip(lower=1e-8)).dropna()

        for cfg_name, cfg_cols in CONFIGS_W96.items():
            it_cols_used = [c for c in cfg_cols if c in X_it_aug_W96.columns]
            tda_cols_used = [c for c in cfg_cols if c in X_tda_full_W96.columns]

            Xi_cfg = pd.concat(
                [X_it_aug_W96[it_cols_used]] +
                ([X_tda_full_W96[tda_cols_used]] if tda_cols_used else []),
                axis=1,
            )

            Xi_cfg = Xi_cfg.loc[:, ~Xi_cfg.columns.duplicated()]

            idx = X_vol.index.intersection(Xi_cfg.index).intersection(y_raw.index)

            valid = (
                X_vol.loc[idx].notna().all(1)
                & Xi_cfg.loc[idx].notna().all(1)
                & y_raw.loc[idx].notna()
            )

            idx = idx[valid]

            if len(idx) < 1000:
                continue

            tr = idx < (split_ts - pd.Timedelta(minutes=30 * PURGE_BARS))
            te = idx >= split_ts

            if tr.sum() < 480 or te.sum() < 100:
                continue

            Xv_tr = X_vol.loc[idx[tr]]
            Xv_te = X_vol.loc[idx[te]]
            Xi_tr = Xi_cfg.loc[idx[tr]]
            Xi_te = Xi_cfg.loc[idx[te]]
            ytr = y_raw.loc[idx[tr]]
            yte = y_raw.loc[idx[te]]

            naive = float(ytr.mean())

            ck = Xv_tr.corrwith(ytr).abs().dropna()
            top3 = ck.nlargest(3).index.tolist()

            mb = _ridge()
            mb.fit(Xv_tr[top3], ytr)

            pb_tr = mb.predict(Xv_tr[top3])
            pb_te = mb.predict(Xv_te[top3])

            res_tr = ytr.values - pb_tr

            mc = _ridge()
            mc.fit(Xi_tr, res_tr)

            pred = pb_te + mc.predict(Xi_te)

            r2_b = oos_r2(yte.values, pb_te, naive)
            r2_c = oos_r2(yte.values, pred, naive)

            rc = float(pd.Series(mc.predict(Xi_te)).corr(pd.Series(yte.values - pb_te)))

            config_rows.append({
                "asset": asset,
                "horizon": h_label,
                "h": h,
                "config": cfg_name,
                "r2_base": round(r2_b, 4),
                "r2_cfg": round(r2_c, 4),
                "delta_r2": round(r2_c - r2_b, 4),
                "res_corr": round(rc, 4),
            })

cfg_df_W96 = pd.DataFrame(config_rows)

print("\nMean ΔR² W96 by config:")
for cfg in CONFIGS_W96:
    sub = cfg_df_W96[cfg_df_W96.config == cfg]
    print(
        f"  {cfg:<15}: ΔR²={sub['delta_r2'].mean():+.4f} "
        f"↑{(sub['delta_r2'] > 0).sum()}/{len(sub)} "
        f"rc={sub['res_corr'].mean():+.4f}"
    )

best_cfg_W96 = cfg_df_W96.groupby("config")["delta_r2"].mean().idxmax()
print(f"\nBest config W96: {best_cfg_W96}")

feat_analysis_W96 = {
    "perm": avg_perm_W96,
    "cfg_df": cfg_df_W96,
    "it_corrs": it_corrs_W96,
    "tda_corrs": tda_corrs_W96,
    "best_cfg": best_cfg_W96,
    "best_cols": CONFIGS_W96[best_cfg_W96],
    "X_it_aug": X_it_aug_W96,
    "X_tda_full": X_tda_full_W96,
    "extra_it_cols": extra_it_cols_W96,
    "feature_window": FEATURE_WINDOW,
}

print("\nfeat_analysis_W96 saved in memory")

FEATURE ANALYSIS W=96
Building extra standalone MI features W96: gaussian
Building extra standalone MI features W96: knn
Building index / entropy / NMI features...
Base IT W96:      25
IT slim W96:      10
Extra IT W96:     10
Total IT W96:     35
Total TDA W96:    26

PERMUTATION IMPORTANCE W=96 ON TRAIN
BTC baseline CV R² = -0.3335
ETH baseline CV R² = -0.5296

Top-15 useful W96 features:
  pairwise_mi_mean                     -0.0367 [IT_slim]
  knn_pairwise_mi_mean                 -0.0263 [IT_extra]
  entropy_index_1y                     -0.0239 [IT_extra]
  kl_AVAX                              -0.0232 [IT_slim]
  coinfo3_mean                         -0.0148 [IT_slim]
  A_E_H1                               -0.0144 [TDA_A]
  gaussian_pairwise_mi_std             -0.0123 [IT_extra]
  BI_tp_H1                             -0.0107 [TDA_BI]
  BI_L1_H1                             -0.0061 [TDA_BI]
  BC_tp_H1                             -0.0047 [TDA_BC]
  knn_pairwise_mi_max                 

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from scipy import stats as sp_stats

from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

SPLIT_DATE = "2025-07-01"
PURGE_BARS = 96

HORIZONS = {
    48:  "1d",
    96:  "2d",
    192: "4d",
    288: "6d",
    336: "7d",
    672: "14d",
}

ASSETS = [a for a in ["BTC","ETH","SOL","BNB","XRP","DOGE","ADA","AVAX"] if a in ret.columns]

VOL_WINDOW = 48
ANN = np.sqrt(365 * 48)

PERM_USEFUL_THRESHOLD = -0.002
TOP_IT = 10
TOP_TDA = 6

def _ridge():
    return Pipeline([
        ("sc", StandardScaler()),
        ("m", RidgeCV(alphas=[0.01, 0.1, 1, 10, 100]))
    ])

def oos_r2(y_true, y_pred, naive_val):
    mse_m = np.mean((y_true - y_pred) ** 2)
    mse_n = np.mean((y_true - naive_val) ** 2)
    return float(1.0 - mse_m / mse_n) if mse_n > 1e-12 else np.nan

def res_corr(residuals_true, residuals_pred):
    return float(pd.Series(residuals_pred).corr(pd.Series(residuals_true)))

def hit_rate_vs_baseline(y_true, pred_base, pred_model):
    e_base = (y_true - pred_base) ** 2
    e_mod = (y_true - pred_model) ** 2
    return float(np.mean(e_mod < e_base))

def dm_test(loss_diff):
    n = len(loss_diff)
    m = loss_diff.mean()
    s = loss_diff.std(ddof=1)
    t = m / (s / np.sqrt(n)) if s > 1e-12 else 0.0
    p = float(sp_stats.t.sf(abs(t), df=n - 1) * 2) if n > 1 else np.nan
    return float(t), float(p)

def sig(p):
    if not np.isfinite(p):
        return ""
    if p < 0.01:
        return "***"
    if p < 0.05:
        return "**"
    if p < 0.10:
        return "*"
    return ""

avg_perm = feat_analysis_W96["perm"]
X_it_all = feat_analysis_W96["X_it_aug"].copy()
X_tda_all = feat_analysis_W96["X_tda_full"].copy()

perm_series = pd.Series(avg_perm).sort_values()

it_candidates = [
    f for f in perm_series.index
    if (f in X_it_all.columns) and (perm_series[f] < PERM_USEFUL_THRESHOLD)
]

tda_candidates = [
    f for f in perm_series.index
    if (f in X_tda_all.columns) and (perm_series[f] < PERM_USEFUL_THRESHOLD)
]

BEST_IT_W96 = it_candidates[:TOP_IT]
BEST_TDA_W96 = tda_candidates[:TOP_TDA]

print("=" * 80)
print("FINAL HOLDOUT EXPERIMENT — W=96")
print("=" * 80)

print("\nSelected IT features:")
for f in BEST_IT_W96:
    print(f"  {f:<36} perm={perm_series[f]:+.4f}")

print("\nSelected TDA features:")
for f in BEST_TDA_W96:
    print(f"  {f:<36} perm={perm_series[f]:+.4f}")

print(f"\nIT count:  {len(BEST_IT_W96)}")
print(f"TDA count: {len(BEST_TDA_W96)}")

split_ts = pd.Timestamp(SPLIT_DATE)
rows = []

for asset in ASSETS:
    r = ret[asset]

    for h, h_label in HORIZONS.items():
        vol_fwd = r.rolling(h).std().shift(-h) * ANN
        y_raw = np.log(vol_fwd.clip(lower=1e-8)).dropna()

        Xi_it = X_it_all[BEST_IT_W96].copy()
        Xi_tda = X_tda_all[BEST_TDA_W96].copy()

        idx = (
            X_vol.index
            .intersection(Xi_it.index)
            .intersection(Xi_tda.index)
            .intersection(y_raw.index)
        )

        valid = (
            X_vol.loc[idx].notna().all(1)
            & Xi_it.loc[idx].notna().all(1)
            & Xi_tda.loc[idx].notna().all(1)
            & y_raw.loc[idx].notna()
        )

        idx = idx[valid]

        if len(idx) < 1000:
            continue

        tr = idx < (split_ts - pd.Timedelta(minutes=30 * PURGE_BARS))
        te = idx >= split_ts

        if tr.sum() < 480 or te.sum() < 100:
            continue

        Xv_tr = X_vol.loc[idx[tr]]
        Xv_te = X_vol.loc[idx[te]]

        Xit_tr = Xi_it.loc[idx[tr]]
        Xit_te = Xi_it.loc[idx[te]]

        Xtd_tr = Xi_tda.loc[idx[tr]]
        Xtd_te = Xi_tda.loc[idx[te]]

        ytr = y_raw.loc[idx[tr]]
        yte = y_raw.loc[idx[te]]

        naive = float(ytr.mean())

        ck = Xv_tr.corrwith(ytr).abs().replace([np.inf, -np.inf], np.nan).dropna()
        top3 = ck.nlargest(3).index.tolist()

        mb = _ridge()
        mb.fit(Xv_tr[top3], ytr)

        pb_tr = mb.predict(Xv_tr[top3])
        pb_te = mb.predict(Xv_te[top3])

        res_tr = ytr.values - pb_tr
        res_te = yte.values - pb_te

        mit = _ridge()
        mit.fit(Xit_tr, res_tr)
        pit_te = mit.predict(Xit_te)
        pred_it = pb_te + pit_te

        mtda = _ridge()
        mtda.fit(Xtd_tr, res_tr)
        ptda_te = mtda.predict(Xtd_te)
        pred_tda = pb_te + ptda_te

        Xc_tr = pd.concat([Xit_tr, Xtd_tr], axis=1)
        Xc_te = pd.concat([Xit_te, Xtd_te], axis=1)

        mct = _ridge()
        mct.fit(Xc_tr, res_tr)
        pct_te = mct.predict(Xc_te)
        pred_all = pb_te + pct_te

        r2_A = oos_r2(yte.values, pb_te, naive)
        r2_B = oos_r2(yte.values, pred_it, naive)
        r2_C = oos_r2(yte.values, pred_tda, naive)
        r2_D = oos_r2(yte.values, pred_all, naive)

        mse_A = np.mean((yte.values - pb_te) ** 2)
        mse_B = np.mean((yte.values - pred_it) ** 2)
        mse_C = np.mean((yte.values - pred_tda) ** 2)
        mse_D = np.mean((yte.values - pred_all) ** 2)

        dB = (yte.values - pb_te) ** 2 - (yte.values - pred_it) ** 2
        dC = (yte.values - pb_te) ** 2 - (yte.values - pred_tda) ** 2
        dD = (yte.values - pb_te) ** 2 - (yte.values - pred_all) ** 2

        _, p_B = dm_test(dB)
        _, p_C = dm_test(dC)
        _, p_D = dm_test(dD)

        rows.append({
            "asset": asset,
            "horizon": h_label,
            "h": h,

            "r2_A": round(r2_A, 4),
            "r2_B": round(r2_B, 4),
            "r2_C": round(r2_C, 4),
            "r2_D": round(r2_D, 4),

            "dR2_B": round(r2_B - r2_A, 4),
            "dR2_C": round(r2_C - r2_A, 4),
            "dR2_D": round(r2_D - r2_A, 4),

            "rc_B": round(res_corr(res_te, pit_te), 4),
            "rc_C": round(res_corr(res_te, ptda_te), 4),
            "rc_D": round(res_corr(res_te, pct_te), 4),

            "mse_ratio_B": round(mse_B / mse_A, 4) if mse_A > 0 else np.nan,
            "mse_ratio_C": round(mse_C / mse_A, 4) if mse_A > 0 else np.nan,
            "mse_ratio_D": round(mse_D / mse_A, 4) if mse_A > 0 else np.nan,

            "hit_B": round(hit_rate_vs_baseline(yte.values, pb_te, pred_it), 4),
            "hit_C": round(hit_rate_vs_baseline(yte.values, pb_te, pred_tda), 4),
            "hit_D": round(hit_rate_vs_baseline(yte.values, pb_te, pred_all), 4),

            "p_B": round(p_B, 4),
            "p_C": round(p_C, 4),
            "p_D": round(p_D, 4),

            "sig_B": sig(p_B),
            "sig_C": sig(p_C),
            "sig_D": sig(p_D),

            "n_train": int(tr.sum()),
            "n_test": int(te.sum()),
        })

df_best_W96 = pd.DataFrame(rows)

print(f"\nRows collected: {len(df_best_W96)}")

print("\n" + "=" * 100)
print("FINAL COMPARISON — W=96")
print("A = Baseline(vol) | B = +best IT | C = +best TDA | D = +best IT + best TDA")
print("=" * 100)

print(
    f"\n  {'asset':<6} {'h':<4}"
    f" {'R²_A':>9} {'R²_B':>9} {'R²_C':>9} {'R²_D':>9}"
    f" {'ΔB':>8} {'ΔC':>8} {'ΔD':>8}  best"
)
print("  " + "-" * 90)

prev = None

for _, row in df_best_W96.sort_values(["asset", "h"]).iterrows():
    if prev and prev != row["asset"]:
        print("  " + "·" * 90)

    prev = row["asset"]

    vals = {"A": row["r2_A"], "B": row["r2_B"], "C": row["r2_C"], "D": row["r2_D"]}
    best = max(vals, key=vals.get)

    mb = "↑" if row["dR2_B"] > 0.005 else ("↓" if row["dR2_B"] < -0.005 else "~")
    mc = "↑" if row["dR2_C"] > 0.005 else ("↓" if row["dR2_C"] < -0.005 else "~")
    md = "↑" if row["dR2_D"] > 0.005 else ("↓" if row["dR2_D"] < -0.005 else "~")

    print(
        f"  {row['asset']:<6} {row['horizon']:<4}"
        f" {row['r2_A']:>+9.4f}"
        f" {row['r2_B']:>+9.4f}"
        f" {row['r2_C']:>+9.4f}"
        f" {row['r2_D']:>+9.4f}"
        f" {row['dR2_B']:>+7.4f}{mb}"
        f" {row['dR2_C']:>+7.4f}{mc}"
        f" {row['dR2_D']:>+7.4f}{md}"
        f"  {best}"
    )

print("\n" + "=" * 75)
print("SUMMARY — W=96")
print("=" * 75)

n = len(df_best_W96)

print(f"""
  Config         mean ΔR²   ΔR²>0    mean rc   MSE<1    hit rate
  ----------------------------------------------------------------
  B (+best IT)   {df_best_W96['dR2_B'].mean():>+8.4f}   {(df_best_W96['dR2_B']>0).sum():>2}/{n}   {df_best_W96['rc_B'].mean():>+7.4f}   {(df_best_W96['mse_ratio_B']<1).sum():>2}/{n}   {df_best_W96['hit_B'].mean():.3f}
  C (+best TDA)  {df_best_W96['dR2_C'].mean():>+8.4f}   {(df_best_W96['dR2_C']>0).sum():>2}/{n}   {df_best_W96['rc_C'].mean():>+7.4f}   {(df_best_W96['mse_ratio_C']<1).sum():>2}/{n}   {df_best_W96['hit_C'].mean():.3f}
  D (+both)      {df_best_W96['dR2_D'].mean():>+8.4f}   {(df_best_W96['dR2_D']>0).sum():>2}/{n}   {df_best_W96['rc_D'].mean():>+7.4f}   {(df_best_W96['mse_ratio_D']<1).sum():>2}/{n}   {df_best_W96['hit_D'].mean():.3f}
""")

asset_summary_W96 = (
    df_best_W96.groupby("asset", as_index=False)
    .agg(
        r2_A=("r2_A", "mean"),
        r2_B=("r2_B", "mean"),
        r2_C=("r2_C", "mean"),
        r2_D=("r2_D", "mean"),
        dR2_B=("dR2_B", "mean"),
        dR2_C=("dR2_C", "mean"),
        dR2_D=("dR2_D", "mean"),
        mse_ratio_D=("mse_ratio_D", "mean"),
    )
)

asset_summary_W96["best_config"] = asset_summary_W96.apply(
    lambda row: max(
        {"A": row["r2_A"], "B": row["r2_B"], "C": row["r2_C"], "D": row["r2_D"]},
        key={"A": row["r2_A"], "B": row["r2_B"], "C": row["r2_C"], "D": row["r2_D"]}.get,
    ),
    axis=1,
)

print("\nASSET SUMMARY — W=96")
print(asset_summary_W96.sort_values("r2_D", ascending=False).to_string(index=False))

final_comparison_best_W96 = df_best_W96.copy()
asset_summary_best_W96 = asset_summary_W96.copy()

best_feature_sets_W96 = {
    "FEATURE_WINDOW": 96,
    "BEST_IT": BEST_IT_W96,
    "BEST_TDA": BEST_TDA_W96,
}

print("\nSaved to memory:")
print("  final_comparison_best_W96")
print("  asset_summary_best_W96")
print("  best_feature_sets_W96")

FINAL HOLDOUT EXPERIMENT — W=96

Selected IT features:
  pairwise_mi_mean                     perm=-0.0367
  knn_pairwise_mi_mean                 perm=-0.0263
  entropy_index_1y                     perm=-0.0239
  kl_AVAX                              perm=-0.0232
  coinfo3_mean                         perm=-0.0148
  gaussian_pairwise_mi_std             perm=-0.0123
  knn_pairwise_mi_max                  perm=-0.0022

Selected TDA features:
  A_E_H1                               perm=-0.0144
  BI_tp_H1                             perm=-0.0107
  BI_L1_H1                             perm=-0.0061
  BC_tp_H1                             perm=-0.0047

IT count:  7
TDA count: 4

Rows collected: 48

FINAL COMPARISON — W=96
A = Baseline(vol) | B = +best IT | C = +best TDA | D = +best IT + best TDA

  asset  h         R²_A      R²_B      R²_C      R²_D       ΔB       ΔC       ΔD  best
  ------------------------------------------------------------------------------------------
  ADA    1d     +0.19

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from scipy import stats as sp_stats

from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mutual_info_score

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

WINDOWS_TO_TEST = [96, 144, 192, 384]
IT_STEP   = 4
IT_BINS   = 10

SPLIT_DATE = "2025-07-01"
PURGE_BARS = 96

HORIZONS = {
    48:  "1d",
    96:  "2d",
    192: "4d",
    288: "6d",
    336: "7d",
    672: "14d",
}

ASSETS = [a for a in ["BTC","ETH","SOL","BNB","XRP","DOGE","ADA","AVAX"]
          if a in ret.columns]

VOL_WINDOW = 48
ANN = np.sqrt(365 * 48)

PERM_USEFUL_THRESHOLD = -0.002
TOP_IT  = 10
TOP_TDA = 6

def _ridge():
    return Pipeline([
        ("sc", StandardScaler()),
        ("m", RidgeCV(alphas=[0.01, 0.1, 1, 10, 100]))
    ])

def oos_r2(y_true, y_pred, naive_val):
    mse_m = np.mean((y_true - y_pred) ** 2)
    mse_n = np.mean((y_true - naive_val) ** 2)
    return float(1.0 - mse_m / mse_n) if mse_n > 1e-12 else np.nan

def dm_test(loss_diff):
    n = len(loss_diff)
    m = loss_diff.mean()
    s = loss_diff.std(ddof=1)
    t = m / (s / np.sqrt(n)) if s > 1e-12 else 0.0
    p = float(sp_stats.t.sf(abs(t), df=n - 1) * 2) if n > 1 else np.nan
    return float(t), float(p)

def sig(p):
    if not np.isfinite(p): return ""
    if p < 0.01: return "***"
    if p < 0.05: return "**"
    if p < 0.10: return "*"
    return ""

def _rank_to_bins(x, bins=10):
    x = np.asarray(x, dtype=float)
    ranks = pd.Series(x).rank(method="average").values
    scaled = np.floor((ranks - 1) / max(1, len(x)) * bins).astype(int)
    return np.clip(scaled, 0, bins - 1)

def _drop_finite(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    return x[mask], y[mask]

def mi_hist_rank(x, y, bins=10):
    x, y = _drop_finite(x, y)
    if len(x) < max(20, bins): return np.nan
    bx = _rank_to_bins(x, bins)
    by = _rank_to_bins(y, bins)
    c = np.histogram2d(bx, by, bins=bins,
                       range=[[0, bins], [0, bins]])[0]
    return float(mutual_info_score(None, None, contingency=c))

def _entropy(x, bins=10):
    x = x[np.isfinite(x)]
    if len(x) < 5: return np.nan
    bx = _rank_to_bins(x, bins)
    counts = np.bincount(bx, minlength=bins).astype(float)
    p = counts / counts.sum()
    p = p[p > 0]
    return float(-np.sum(p * np.log(p)))

def _joint_entropy(xs, bins=10):
    if any(len(x) < 5 for x in xs): return np.nan
    multi = np.stack([_rank_to_bins(x, bins) for x in xs], axis=1)
    codes = multi @ (bins ** np.arange(len(xs)))
    counts = np.bincount(codes,
                         minlength=bins**len(xs)).astype(float)
    p = counts / counts.sum()
    p = p[p > 0]
    return float(-np.sum(p * np.log(p)))

def build_it_features_for_window(ret_df, assets, window, step, bins):
    """
    Builds the base IT features (pairwise MI, KL, coinfo3, coinfo4)
    for the given window size.
    Returns a DataFrame with the same columns as X_it_aug,
    but computed with a different window.
    """
    avail  = [a for a in assets if a in ret_df.columns]
    idx    = ret_df.index
    n      = len(idx)
    starts = np.arange(0, n - window + 1, step)

    pairs   = [(avail[i], avail[j])
                for i in range(len(avail))
                for j in range(i + 1, len(avail))]
    triples = [(avail[i], avail[j], avail[k])
                for i in range(len(avail))
                for j in range(i + 1, len(avail))
                for k in range(j + 1, len(avail))]
    quads   = [(avail[i], avail[j], avail[k], avail[l])
                for i in range(len(avail))
                for j in range(i + 1, len(avail))
                for k in range(j + 1, len(avail))
                for l in range(k + 1, len(avail))]

    cols = (
        ["pairwise_mi_mean", "pairwise_mi_std",
         "pairwise_mi_max",  "pairwise_mi_min"] +
        [f"kl_{a}" for a in avail] +
        ["coinfo3_mean", "coinfo3_std",
         "coinfo3_max",  "coinfo3_min"] +
        ["coinfo4_mean", "coinfo4_std",
         "coinfo4_max",  "coinfo4_min"]
    )
    out = pd.DataFrame(np.nan, index=idx, columns=cols)

    print(f"    Building IT W={window}: {len(starts)} windows...")

    for s in starts:
        e   = s + window
        sub = ret_df.iloc[s:e]

        mi_arr = np.array(
            [v for v in [mi_hist_rank(sub[a].values,
                                      sub[b].values, bins)
                         for a, b in pairs]
             if np.isfinite(v)], dtype=float)
        if len(mi_arr):
            out.at[idx[e-1], "pairwise_mi_mean"] = mi_arr.mean()
            out.at[idx[e-1], "pairwise_mi_std"]  = mi_arr.std(ddof=0)
            out.at[idx[e-1], "pairwise_mi_max"]  = mi_arr.max()
            out.at[idx[e-1], "pairwise_mi_min"]  = mi_arr.min()

        if s >= window:
            prev = ret_df.iloc[s - window:s]
            for a in avail:
                xc = sub[a].values; xp = prev[a].values
                if len(xc) < 5 or len(xp) < 5: continue
                pc = (np.bincount(_rank_to_bins(xc, bins),
                                  minlength=bins).astype(float) + 1e-8)
                pp = (np.bincount(_rank_to_bins(xp, bins),
                                  minlength=bins).astype(float) + 1e-8)
                pc /= pc.sum(); pp /= pp.sum()
                out.at[idx[e-1], f"kl_{a}"] = float(
                    np.sum(pc * np.log(pc / pp)))

        ci3_vals = []
        for (a, b, c) in triples:
            xa = sub[a].values
            xb = sub[b].values
            xc_ = sub[c].values
            vs = [_entropy(xa), _entropy(xb), _entropy(xc_),
                  _joint_entropy([xa, xb]),
                  _joint_entropy([xa, xc_]),
                  _joint_entropy([xb, xc_]),
                  _joint_entropy([xa, xb, xc_])]
            if all(v is not None and np.isfinite(v) for v in vs):
                h_a,h_b,h_c,h_ab,h_ac,h_bc,h_abc = vs
                ci3_vals.append(
                    h_a+h_b+h_c - h_ab-h_ac-h_bc + h_abc)
        if ci3_vals:
            c3 = np.array(ci3_vals)
            out.at[idx[e-1], "coinfo3_mean"] = c3.mean()
            out.at[idx[e-1], "coinfo3_std"]  = c3.std(ddof=0)
            out.at[idx[e-1], "coinfo3_max"]  = c3.max()
            out.at[idx[e-1], "coinfo3_min"]  = c3.min()

        ci4_vals = []
        for (a, b, c, d) in quads:
            xs_ = [sub[x].values for x in [a, b, c, d]]
            h1 = [_entropy(x) for x in xs_]
            h2 = [_joint_entropy([xs_[i], xs_[j]])
                  for i in range(4)
                  for j in range(i+1, 4)]
            h3 = [_joint_entropy([xs_[i], xs_[j], xs_[k]])
                  for i in range(4)
                  for j in range(i+1, 4)
                  for k in range(j+1, 4)]
            h4 = _joint_entropy(xs_)
            if all(v is not None and np.isfinite(v)
                   for v in h1+h2+h3+[h4]):
                ci4_vals.append(sum(h1)-sum(h2)+sum(h3)-h4)
        if ci4_vals:
            c4 = np.array(ci4_vals)
            out.at[idx[e-1], "coinfo4_mean"] = c4.mean()
            out.at[idx[e-1], "coinfo4_std"]  = c4.std(ddof=0)
            out.at[idx[e-1], "coinfo4_max"]  = c4.max()
            out.at[idx[e-1], "coinfo4_min"]  = c4.min()

    return out.ffill()

print("Loading TDA features...")

if "feat_analysis" in dir() and "X_tda_full" in feat_analysis:
    X_tda_all = feat_analysis["X_tda_full"].copy()
elif "X_tda_full" in dir():
    X_tda_all = X_tda_full.copy()
else:
    tda_frames = [tda_A.add_prefix("A_")]
    for mode in ["I", "II", "C"]:
        d = tda_B[mode]
        df_m = pd.DataFrame({
            f"B{mode}_tp_H0":  d["tp_H0"],
            f"B{mode}_ent_H0": d["ent_H0"],
            f"B{mode}_tp_H1":  d["tp_H1"],
            f"B{mode}_ent_H1": d["ent_H1"],
            f"B{mode}_L1_H0":  d["L1_H0"],
            f"B{mode}_L1_H1":  d["L1_H1"],
        }, index=pd.DatetimeIndex(d["times"]))
        tda_frames.append(df_m)
    X_tda_all = (pd.concat(tda_frames, axis=1)
                   .sort_index()
                   .reindex(ret.index, method="ffill"))

print(f"  TDA shape: {X_tda_all.shape}")

if "best_feature_sets" in dir():
    BEST_TDA = list(best_feature_sets.get("BEST_TDA", []))
else:
    BEST_TDA = ["BI_L1_H0", "BC_ent_H1", "BII_tp_H1",
                "BII_L1_H1", "A_T_H0", "A_E_H0"]

BEST_TDA = [c for c in BEST_TDA if c in X_tda_all.columns]
print(f"  Using {len(BEST_TDA)} TDA features: {BEST_TDA}")

def run_holdout_for_window(X_it_w, best_it, best_tda, window_label):
    """
    Runs the A/B/C/D comparison for one set of IT features.
    Returns a DataFrame with the results.
    """
    split_ts = pd.Timestamp(SPLIT_DATE)
    rows = []

    for asset in ASSETS:
        r = ret[asset]
        Xi_it  = X_it_w[best_it].copy()
        Xi_tda = X_tda_all[best_tda].copy()

        for h, h_label in HORIZONS.items():
            vol_fwd = r.rolling(h).std().shift(-h) * ANN
            y_raw   = np.log(vol_fwd.clip(lower=1e-8)).dropna()

            idx = (X_vol.index
                   .intersection(Xi_it.index)
                   .intersection(Xi_tda.index)
                   .intersection(y_raw.index))
            valid = (X_vol.loc[idx].notna().all(1)
                     & Xi_it.loc[idx].notna().all(1)
                     & Xi_tda.loc[idx].notna().all(1)
                     & y_raw.loc[idx].notna())
            idx = idx[valid]

            if len(idx) < 1000: continue

            tr = idx < (split_ts - pd.Timedelta(minutes=30 * PURGE_BARS))
            te = idx >= split_ts
            if tr.sum() < 480 or te.sum() < 100: continue

            Xv_tr  = X_vol.loc[idx[tr]]; Xv_te  = X_vol.loc[idx[te]]
            Xit_tr = Xi_it.loc[idx[tr]]; Xit_te = Xi_it.loc[idx[te]]
            Xtd_tr = Xi_tda.loc[idx[tr]]; Xtd_te = Xi_tda.loc[idx[te]]
            ytr = y_raw.loc[idx[tr]]; yte = y_raw.loc[idx[te]]
            naive = float(ytr.mean())

            ck   = (Xv_tr.corrwith(ytr).abs()
                    .replace([np.inf, -np.inf], np.nan).dropna())
            top3 = ck.nlargest(3).index.tolist()
            mb = _ridge(); mb.fit(Xv_tr[top3], ytr)
            pb_tr = mb.predict(Xv_tr[top3])
            pb_te = mb.predict(Xv_te[top3])
            res_tr = ytr.values - pb_tr
            res_te = yte.values - pb_te

            mit = _ridge(); mit.fit(Xit_tr, res_tr)
            pit_te  = mit.predict(Xit_te)
            pred_it = pb_te + pit_te

            mtda = _ridge(); mtda.fit(Xtd_tr, res_tr)
            ptda_te  = mtda.predict(Xtd_te)
            pred_tda = pb_te + ptda_te

            Xc_tr = pd.concat([Xit_tr, Xtd_tr], axis=1)
            Xc_te = pd.concat([Xit_te, Xtd_te], axis=1)
            mct = _ridge(); mct.fit(Xc_tr, res_tr)
            pct_te   = mct.predict(Xc_te)
            pred_all = pb_te + pct_te

            r2_A = oos_r2(yte.values, pb_te,    naive)
            r2_B = oos_r2(yte.values, pred_it,  naive)
            r2_C = oos_r2(yte.values, pred_tda, naive)
            r2_D = oos_r2(yte.values, pred_all, naive)

            dD = ((yte.values - pb_te)**2
                  - (yte.values - pred_all)**2)
            _, p_D = dm_test(dD)

            rows.append({
                "window":  window_label,
                "asset":   asset,
                "horizon": h_label,
                "h":       h,
                "r2_A":   round(r2_A, 4),
                "r2_B":   round(r2_B, 4),
                "r2_C":   round(r2_C, 4),
                "r2_D":   round(r2_D, 4),
                "dR2_B":  round(r2_B - r2_A, 4),
                "dR2_C":  round(r2_C - r2_A, 4),
                "dR2_D":  round(r2_D - r2_A, 4),
                "p_D":    round(p_D, 4),
                "sig_D":  sig(p_D),
                "n_train": int(tr.sum()),
                "n_test":  int(te.sum()),
            })

    return pd.DataFrame(rows)

print("\n" + "=" * 70)
print("WINDOW SENSITIVITY ANALYSIS")
print(f"Windows: {WINDOWS_TO_TEST} bars")
print("=" * 70)

all_results = []
window_it_features = {}

for w in WINDOWS_TO_TEST:
    w_label = f"W={w}"
    w_days  = w / 48
    print(f"\n{'='*60}")
    print(f"  Window: {w} bars (~{w_days:.1f} days)")
    print(f"{'='*60}")

    X_it_w = build_it_features_for_window(
        ret, ASSETS, w, IT_STEP, IT_BINS)
    print(f"  IT features built: {X_it_w.shape}")

    if "feat_analysis" in dir():
        avg_perm_src = feat_analysis["perm"]
    else:
        avg_perm_src = avg_perm

    perm_series = pd.Series(avg_perm_src).sort_values()
    best_it_w = [
        f for f in perm_series.index
        if (f in X_it_w.columns)
        and (perm_series[f] < PERM_USEFUL_THRESHOLD)
    ][:TOP_IT]

    if len(best_it_w) == 0:
        best_it_w = [
            f for f in perm_series.index
            if f in X_it_w.columns
        ][:6]
        print(f"  WARNING: no features passed threshold, "
              f"using top-{len(best_it_w)} by perm score")

    print(f"  Selected IT features ({len(best_it_w)}): {best_it_w}")

    window_it_features[w] = (X_it_w, best_it_w)

    df_w = run_holdout_for_window(
        X_it_w, best_it_w, BEST_TDA, w_label)
    all_results.append(df_w)

    n = len(df_w)
    if n > 0:
        print(f"\n  Results ({n} tasks):")
        print(f"    B (+IT):  ΔR²={df_w['dR2_B'].mean():>+.4f}  "
              f"pos={(df_w['dR2_B']>0).sum():>2}/{n}")
        print(f"    C (+TDA): ΔR²={df_w['dR2_C'].mean():>+.4f}  "
              f"pos={(df_w['dR2_C']>0).sum():>2}/{n}")
        print(f"    D (+all): ΔR²={df_w['dR2_D'].mean():>+.4f}  "
              f"pos={(df_w['dR2_D']>0).sum():>2}/{n}")
    else:
        print("  WARNING: no rows collected for this window.")

df_window = pd.concat(all_results, ignore_index=True)
print(f"\nTotal rows: {len(df_window)}")

print("\n" + "=" * 70)
print("WINDOW SENSITIVITY SUMMARY")
print("=" * 70)

print(f"\n  {'Window':<10} {'bars':>6} {'days':>5}"
      f" {'ΔR²_B':>8} {'ΔR²_C':>8} {'ΔR²_D':>8}"
      f" {'B>0':>6} {'C>0':>6} {'D>0':>6}")
print("  " + "-" * 65)

for w in WINDOWS_TO_TEST:
    w_label = f"W={w}"
    sub = df_window[df_window.window == w_label]
    if sub.empty: continue
    n = len(sub)
    print(f"  {w_label:<10} {w:>6} {w/48:>5.1f}"
          f" {sub['dR2_B'].mean():>+8.4f}"
          f" {sub['dR2_C'].mean():>+8.4f}"
          f" {sub['dR2_D'].mean():>+8.4f}"
          f" {(sub['dR2_B']>0).sum():>4}/{n}"
          f" {(sub['dR2_C']>0).sum():>4}/{n}"
          f" {(sub['dR2_D']>0).sum():>4}/{n}")

print(f"\n  Asset-level ΔR²_D by window:")
print(f"  {'Asset':<6}", end="")
for w in WINDOWS_TO_TEST:
    print(f"  {'W='+str(w):>10}", end="")
print()
print("  " + "-" * (6 + 12 * len(WINDOWS_TO_TEST)))

for asset in ASSETS:
    print(f"  {asset:<6}", end="")
    for w in WINDOWS_TO_TEST:
        sub = df_window[(df_window.window == f"W={w}")
                        & (df_window.asset == asset)]
        val = sub["dR2_D"].mean() if not sub.empty else np.nan
        if np.isfinite(val):
            print(f"  {val:>+10.4f}", end="")
        else:
            print(f"  {'N/A':>10}", end="")
    print()

print("\nBuilding window sensitivity plots...")

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
w_labels  = [f"W={w}" for w in WINDOWS_TO_TEST]
w_days    = [f"{w/48:.1f}d" for w in WINDOWS_TO_TEST]
colors    = {"B": "#2196F3", "C": "#FF9800", "D": "#4CAF50"}

ax = axes[0]
for cfg in ["B", "C", "D"]:
    vals = [df_window[df_window.window == wl][f"dR2_{cfg}"].mean()
            for wl in w_labels]
    ax.plot(w_days, vals, "o-", color=colors[cfg],
            lw=2, ms=8, label=f"Config {cfg}")
ax.axhline(0, color="black", lw=0.8, ls=":")
ax.set_xlabel("IT window size")
ax.set_ylabel("Mean ΔR² vs baseline A")
ax.set_title("Mean ΔR² by IT window\n(all assets, all horizons)", fontsize=10)
ax.legend(fontsize=9)
ax.grid(alpha=0.25)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax = axes[1]
for cfg in ["B", "C", "D"]:
    fracs = []
    for wl in w_labels:
        sub = df_window[df_window.window == wl]
        n   = len(sub)
        fracs.append((sub[f"dR2_{cfg}"] > 0).sum() / n if n > 0 else np.nan)
    ax.plot(w_days, fracs, "o-", color=colors[cfg],
            lw=2, ms=8, label=f"Config {cfg}")
ax.axhline(0.5, color="gray", lw=0.8, ls=":", alpha=0.5)
ax.set_xlabel("IT window size")
ax.set_ylabel("Fraction of tasks with ΔR² > 0")
ax.set_title("Hit rate by IT window\n(fraction of tasks improved)", fontsize=10)
ax.legend(fontsize=9)
ax.grid(alpha=0.25)
ax.set_ylim(0, 1)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax = axes[2]
heat_data = np.full((len(ASSETS), len(WINDOWS_TO_TEST)), np.nan)
for j, w in enumerate(WINDOWS_TO_TEST):
    wl = f"W={w}"
    for i, asset in enumerate(ASSETS):
        sub = df_window[(df_window.window == wl)
                        & (df_window.asset == asset)]
        if not sub.empty:
            heat_data[i, j] = sub["dR2_D"].mean()

im = ax.imshow(heat_data, cmap="RdYlGn", vmin=-0.15, vmax=0.15,
               aspect="auto")
ax.set_xticks(range(len(WINDOWS_TO_TEST)))
ax.set_xticklabels(w_days)
ax.set_yticks(range(len(ASSETS)))
ax.set_yticklabels(ASSETS)
ax.set_xlabel("IT window size")
ax.set_title("ΔR²_D by asset × window\n(green = better than baseline)", fontsize=10)

for i in range(len(ASSETS)):
    for j in range(len(WINDOWS_TO_TEST)):
        val = heat_data[i, j]
        if np.isfinite(val):
            col = "white" if abs(val) > 0.08 else "black"
            ax.text(j, i, f"{val:+.3f}", ha="center", va="center",
                    fontsize=8, color=col)

plt.colorbar(im, ax=ax, label="ΔR²_D vs baseline A")

plt.suptitle(
    f"Window sensitivity analysis: IT features computed with "
    f"W ∈ {{{', '.join(str(w) for w in WINDOWS_TO_TEST)}}} bars\n"
    f"TDA features unchanged (W=192). Holdout {SPLIT_DATE} →",
    fontsize=12, y=1.01
)
plt.tight_layout()
plt.savefig("fig_window_sensitivity.png", bbox_inches="tight", dpi=150)
plt.savefig("fig_window_sensitivity.pdf", bbox_inches="tight", dpi=150)
plt.show()
print("Saved: fig_window_sensitivity.png / .pdf")

window_sensitivity_results = df_window.copy()
print(f"\nwindow_sensitivity_results saved in memory ({len(df_window)} rows)")

Loading TDA features...
  TDA shape: (46743, 26)
  Using 6 TDA features: ['BI_L1_H0', 'BC_ent_H1', 'BII_tp_H1', 'BII_L1_H1', 'A_T_H0', 'A_E_H0']

WINDOW SENSITIVITY ANALYSIS
Windows: [96, 144, 192, 384] bars

  Window: 96 bars (~2.0 days)
    Building IT W=96: 11662 windows...


In [ ]:
df_window = window_sensitivity_results.copy()
WINDOWS_TO_TEST = sorted(df_window["window"].str.replace("W=","").astype(int).unique().tolist())

print("\n" + "=" * 70)
print("WINDOW SENSITIVITY SUMMARY")
print("=" * 70)

print(f"\n  {'Window':<10} {'bars':>6} {'days':>5}"
      f" {'ΔR²_B':>8} {'ΔR²_C':>8} {'ΔR²_D':>8}"
      f" {'B>0':>6} {'C>0':>6} {'D>0':>6}")
print("  " + "-" * 65)

for w in WINDOWS_TO_TEST:
    w_label = f"W={w}"
    sub = df_window[df_window.window == w_label]
    if sub.empty: continue
    n = len(sub)
    print(f"  {w_label:<10} {w:>6} {w/48:>5.1f}"
          f" {sub['dR2_B'].mean():>+8.4f}"
          f" {sub['dR2_C'].mean():>+8.4f}"
          f" {sub['dR2_D'].mean():>+8.4f}"
          f" {(sub['dR2_B']>0).sum():>4}/{n}"
          f" {(sub['dR2_C']>0).sum():>4}/{n}"
          f" {(sub['dR2_D']>0).sum():>4}/{n}")

print(f"\n  Asset-level ΔR²_D by window:")
ASSETS_order = df_window["asset"].unique().tolist()
print(f"  {'Asset':<6}", end="")
for w in WINDOWS_TO_TEST:
    print(f"  {'W='+str(w):>10}", end="")
print()
print("  " + "-" * (6 + 12 * len(WINDOWS_TO_TEST)))

for asset in ASSETS_order:
    print(f"  {asset:<6}", end="")
    for w in WINDOWS_TO_TEST:
        sub = df_window[(df_window.window == f"W={w}")
                        & (df_window.asset == asset)]
        val = sub["dR2_D"].mean() if not sub.empty else float("nan")
        if val == val:
            print(f"  {val:>+10.4f}", end="")
        else:
            print(f"  {'N/A':>10}", end="")
    print()

print(f"\n  Asset-level ΔR²_C by window:")
print(f"  {'Asset':<6}", end="")
for w in WINDOWS_TO_TEST:
    print(f"  {'W='+str(w):>10}", end="")
print()
print("  " + "-" * (6 + 12 * len(WINDOWS_TO_TEST)))

for asset in ASSETS_order:
    print(f"  {asset:<6}", end="")
    for w in WINDOWS_TO_TEST:
        sub = df_window[(df_window.window == f"W={w}")
                        & (df_window.asset == asset)]
        val = sub["dR2_C"].mean() if not sub.empty else float("nan")
        if val == val:
            print(f"  {val:>+10.4f}", end="")
        else:
            print(f"  {'N/A':>10}", end="")
    print()

print(f"\n  Horizon-level ΔR²_D (mean over assets) by window:")
HORIZONS_order = ["1d","2d","4d","6d","7d","14d"]
print(f"  {'Horizon':<8}", end="")
for w in WINDOWS_TO_TEST:
    print(f"  {'W='+str(w):>10}", end="")
print()
print("  " + "-" * (8 + 12 * len(WINDOWS_TO_TEST)))

for h_label in HORIZONS_order:
    sub_h = df_window[df_window.horizon == h_label]
    if sub_h.empty: continue
    print(f"  {h_label:<8}", end="")
    for w in WINDOWS_TO_TEST:
        sub = sub_h[sub_h.window == f"W={w}"]
        val = sub["dR2_D"].mean() if not sub.empty else float("nan")
        if val == val:
            print(f"  {val:>+10.4f}", end="")
        else:
            print(f"  {'N/A':>10}", end="")
    print()

print(f"\n  Total rows in df_window: {len(df_window)}")
print(f"  Windows found: {[f'W={w}' for w in WINDOWS_TO_TEST]}")
print(f"  Assets found:  {ASSETS_order}")


WINDOW SENSITIVITY SUMMARY

  Window       bars  days    ΔR²_B    ΔR²_C    ΔR²_D    B>0    C>0    D>0
  -----------------------------------------------------------------
  W=96           96   2.0  +0.0204  +0.0536  +0.0850   35/48   32/48   33/48
  W=144         144   3.0  +0.0031  +0.0536  +0.0619   30/48   32/48   30/48
  W=192         192   4.0  -0.0178  +0.0536  +0.0459   25/48   32/48   31/48
  W=384         384   8.0  -0.0401  +0.0536  +0.0146   26/48   32/48   26/48

  Asset-level ΔR²_D by window:
  Asset         W=96       W=144       W=192       W=384
  ------------------------------------------------------
  BTC        -0.0618     -0.0578     -0.0543     -0.0207
  ETH        +0.2298     +0.2219     +0.2094     +0.1478
  SOL        -0.0274     -0.0416     -0.0487     -0.0274
  BNB        -0.0718     -0.0882     -0.1031     -0.1491
  XRP        +0.0908     +0.0744     +0.0728     +0.0836
  DOGE       +0.1922     +0.1605     +0.1452     +0.1153
  ADA        +0.2107     +0.1887 

In [ ]:
df_window = window_sensitivity_results.copy()
WINDOWS_TO_TEST = sorted(
    df_window["window"].str.replace("W=","").astype(int).unique().tolist()
)
HORIZONS_order = ["1d","2d","4d","6d","7d","14d"]
ASSETS_order   = ["BTC","ETH","SOL","BNB","XRP","DOGE","ADA","AVAX"]

def sig(p):
    if not (p == p): return ""
    if p < 0.01: return "***"
    if p < 0.05: return "**"
    if p < 0.10: return "*"
    return ""

for w in WINDOWS_TO_TEST:
    w_label = f"W={w}"
    sub_w   = df_window[df_window.window == w_label]
    n       = len(sub_w)

    print("\n" + "=" * 100)
    print(f"WINDOW {w_label}  (~{w/48:.1f} days) — {n} tasks")
    print("A = vol baseline | B = +IT | C = +TDA | D = +IT+TDA")
    print("=" * 100)

    print(
        f"\n  {'asset':<6} {'h':<4}"
        f" {'R²_A':>9} {'R²_B':>9} {'R²_C':>9} {'R²_D':>9}"
        f" {'ΔB':>8} {'ΔC':>8} {'ΔD':>8}  sig_D  best"
    )
    print("  " + "-" * 95)

    prev = None
    for _, row in sub_w.sort_values(["asset","h"]).iterrows():
        if prev and prev != row["asset"]:
            print("  " + "·" * 95)
        prev = row["asset"]

        vals = {
            "A": row["r2_A"], "B": row["r2_B"],
            "C": row["r2_C"], "D": row["r2_D"],
        }
        best = max(vals, key=vals.get)

        mb = "↑" if row["dR2_B"] >  0.005 else ("↓" if row["dR2_B"] < -0.005 else "~")
        mc = "↑" if row["dR2_C"] >  0.005 else ("↓" if row["dR2_C"] < -0.005 else "~")
        md = "↑" if row["dR2_D"] >  0.005 else ("↓" if row["dR2_D"] < -0.005 else "~")

        print(
            f"  {row['asset']:<6} {row['horizon']:<4}"
            f" {row['r2_A']:>+9.4f}"
            f" {row['r2_B']:>+9.4f}"
            f" {row['r2_C']:>+9.4f}"
            f" {row['r2_D']:>+9.4f}"
            f" {row['dR2_B']:>+7.4f}{mb}"
            f" {row['dR2_C']:>+7.4f}{mc}"
            f" {row['dR2_D']:>+7.4f}{md}"
            f"  {row['sig_D']:>4}   {best}"
        )

    print(f"\n  Asset summary (mean over horizons):")
    print(
        f"  {'asset':<6}"
        f" {'R²_A':>9} {'R²_B':>9} {'R²_C':>9} {'R²_D':>9}"
        f" {'ΔB':>8} {'ΔC':>8} {'ΔD':>8}  best"
    )
    print("  " + "-" * 80)

    for asset in ASSETS_order:
        g = sub_w[sub_w.asset == asset]
        if g.empty: continue
        vals = {
            "A": g["r2_A"].mean(), "B": g["r2_B"].mean(),
            "C": g["r2_C"].mean(), "D": g["r2_D"].mean(),
        }
        best = max(vals, key=vals.get)
        print(
            f"  {asset:<6}"
            f" {g['r2_A'].mean():>+9.4f}"
            f" {g['r2_B'].mean():>+9.4f}"
            f" {g['r2_C'].mean():>+9.4f}"
            f" {g['r2_D'].mean():>+9.4f}"
            f" {g['dR2_B'].mean():>+8.4f}"
            f" {g['dR2_C'].mean():>+8.4f}"
            f" {g['dR2_D'].mean():>+8.4f}"
            f"  {best}"
        )

    print(f"\n  Horizon summary (mean over assets):")
    print(
        f"  {'h':<4}"
        f" {'R²_A':>9} {'R²_B':>9} {'R²_C':>9} {'R²_D':>9}"
        f" {'ΔB':>8} {'ΔC':>8} {'ΔD':>8}"
    )
    print("  " + "-" * 72)

    for h_label in HORIZONS_order:
        g = sub_w[sub_w.horizon == h_label]
        if g.empty: continue
        print(
            f"  {h_label:<4}"
            f" {g['r2_A'].mean():>+9.4f}"
            f" {g['r2_B'].mean():>+9.4f}"
            f" {g['r2_C'].mean():>+9.4f}"
            f" {g['r2_D'].mean():>+9.4f}"
            f" {g['dR2_B'].mean():>+8.4f}"
            f" {g['dR2_C'].mean():>+8.4f}"
            f" {g['dR2_D'].mean():>+8.4f}"
        )

    print(f"\n  Overall: "
          f"ΔR²_B={sub_w['dR2_B'].mean():>+.4f} "
          f"({(sub_w['dR2_B']>0).sum()}/{n})  "
          f"ΔR²_C={sub_w['dR2_C'].mean():>+.4f} "
          f"({(sub_w['dR2_C']>0).sum()}/{n})  "
          f"ΔR²_D={sub_w['dR2_D'].mean():>+.4f} "
          f"({(sub_w['dR2_D']>0).sum()}/{n})")


WINDOW W=96  (~2.0 days) — 48 tasks
A = vol baseline | B = +IT | C = +TDA | D = +IT+TDA

  asset  h         R²_A      R²_B      R²_C      R²_D       ΔB       ΔC       ΔD  sig_D  best
  -----------------------------------------------------------------------------------------------
  ADA    1d     +0.2670   +0.3056   +0.3490   +0.3989 +0.0387↑ +0.0820↑ +0.1320↑   ***   D
  ADA    2d     +0.2413   +0.2880   +0.3515   +0.4171 +0.0467↑ +0.1102↑ +0.1758↑   ***   D
  ADA    4d     +0.1948   +0.2415   +0.3151   +0.3658 +0.0467↑ +0.1203↑ +0.1710↑   ***   D
  ADA    6d     +0.1330   +0.1986   +0.2573   +0.3520 +0.0656↑ +0.1243↑ +0.2190↑   ***   D
  ADA    7d     +0.1210   +0.1872   +0.2543   +0.3592 +0.0662↑ +0.1333↑ +0.2382↑   ***   D
  ADA    14d    -0.0990   -0.1195   +0.1013   +0.2291 -0.0205↓ +0.2003↑ +0.3281↑   ***   D
  ·······························································································
  AVAX   1d     +0.1260   +0.1793   +0.1651   +0.2542 +0.0533↑ +0.0391↑ +0

## 7. Comparison with HAR-RV and GARCH(1,1) at W = 96
HAR-RV is better than the augmented configurations for BTC, SOL and BNB; D beats HAR-RV for ETH, DOGE, ADA and AVAX, and C for XRP.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from scipy import stats as sp_stats

from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

try:
    from arch import arch_model
    HAS_ARCH = True
except ImportError:
    print("arch package not found. Install with: pip install arch")
    HAS_ARCH = False

SPLIT_DATE = "2025-07-01"
PURGE_BARS = 96

HORIZONS = {
    48:  "1d",
    96:  "2d",
    192: "4d",
    288: "6d",
    336: "7d",
    672: "14d",
}

ASSETS = [a for a in ["BTC","ETH","SOL","BNB","XRP","DOGE","ADA","AVAX"] if a in ret.columns]

VOL_WINDOW = 48
ANN = np.sqrt(365 * 48)

PERM_USEFUL_THRESHOLD = -0.002
TOP_IT  = 10
TOP_TDA = 6

GARCH_P = 1
GARCH_Q = 1

def _ridge():
    return Pipeline([
        ("sc", StandardScaler()),
        ("m", RidgeCV(alphas=[0.01, 0.1, 1, 10, 100]))
    ])

def oos_r2(y_true, y_pred, naive_val):
    mse_m = np.mean((y_true - y_pred) ** 2)
    mse_n = np.mean((y_true - naive_val) ** 2)
    return float(1.0 - mse_m / mse_n) if mse_n > 1e-12 else np.nan

def res_corr(residuals_true, residuals_pred):
    return float(pd.Series(residuals_pred).corr(pd.Series(residuals_true)))

def hit_rate_vs_baseline(y_true, pred_base, pred_model):
    e_base = (y_true - pred_base) ** 2
    e_mod  = (y_true - pred_model) ** 2
    return float(np.mean(e_mod < e_base))

def dm_test(loss_diff):
    n = len(loss_diff)
    m = loss_diff.mean()
    s = loss_diff.std(ddof=1)
    t = m / (s / np.sqrt(n)) if s > 1e-12 else 0.0
    p = float(sp_stats.t.sf(abs(t), df=n - 1) * 2) if n > 1 else np.nan
    return float(t), float(p)

def sig(p):
    if not np.isfinite(p): return ""
    if p < 0.01: return "***"
    if p < 0.05: return "**"
    if p < 0.10: return "*"
    return ""

def build_har_features(ret_series):
    rv_d = ret_series.rolling(48).std()   * ANN
    rv_w = ret_series.rolling(240).std()  * ANN
    rv_m = ret_series.rolling(1056).std() * ANN

    return pd.DataFrame({
        "har_d": np.log(rv_d.clip(lower=1e-8)),
        "har_w": np.log(rv_w.clip(lower=1e-8)),
        "har_m": np.log(rv_m.clip(lower=1e-8)),
    })

def fit_garch_forecast(r_train, r_test, h, ann_factor):
    """
    Fits GARCH(1,1) on r_train and predicts the h-step
    conditional volatility for each point of r_test.

    Returns log(annualized_vol_forecast) for each test point.
    Returns NaN if GARCH does not converge.
    """
    if not HAS_ARCH:
        return np.full(len(r_test), np.nan)

    try:
        scale = 100.0
        r_scaled = r_train.values * scale

        am = arch_model(r_scaled, vol="Garch", p=GARCH_P, q=GARCH_Q,
                        mean="Zero", dist="normal")
        res = am.fit(disp="off", show_warning=False)

        fc = res.forecast(horizon=h, reindex=False)
        sigma_h = np.sqrt(fc.variance.iloc[-1].mean()) / scale * ann_factor

        log_sigma = float(np.log(max(sigma_h, 1e-8)))
        return np.full(len(r_test), log_sigma)

    except Exception:
        return np.full(len(r_test), np.nan)

def build_garch_forecasts(ret_series, split_ts, h, ann_factor,
                           purge_bars=PURGE_BARS):
    """
    Builds GARCH forecasts for the test period.
    Train = everything before split_ts - purge_bars.
    Returns a Series with log(vol_forecast) over the test period.
    """
    cutoff = split_ts - pd.Timedelta(minutes=30 * purge_bars)
    r_train = ret_series[ret_series.index < cutoff].dropna()
    r_test  = ret_series[ret_series.index >= split_ts].dropna()

    if len(r_train) < 200 or len(r_test) == 0:
        return pd.Series(np.nan, index=r_test.index)

    preds = fit_garch_forecast(r_train, r_test, h, ann_factor)
    return pd.Series(preds, index=r_test.index)

print("Building GARCH(1,1) forecasts for all assets and horizons...")
split_ts_global = pd.Timestamp(SPLIT_DATE)

garch_forecasts = {}   # {(asset, h): Series on test index}
for asset in ASSETS:
    for h in HORIZONS:
        key = (asset, h)
        garch_forecasts[key] = build_garch_forecasts(
            ret[asset], split_ts_global, h, ANN
        )
        n_valid = garch_forecasts[key].notna().sum()
        print(f"  {asset} h={HORIZONS[h]}: {n_valid} test points")

print("  GARCH forecasts done.")

if "window_it_features" in dir() and 96 in window_it_features:
    X_it_all, BEST_IT_96 = window_it_features[96]
    X_it_all = X_it_all.copy()
    print("Loaded IT features with W=96 from window_it_features[96]")
else:
    raise ValueError(
        "window_it_features[96] not found. "
        "Run window sensitivity script first."
    )

if "feat_analysis" in dir():
    avg_perm  = feat_analysis["perm"]
    if "X_tda_full" not in feat_analysis:
        raise ValueError("X_tda_full missing in feat_analysis.")
    X_tda_all = feat_analysis["X_tda_full"].copy()
else:
    if "avg_perm" not in dir():
        raise ValueError("avg_perm not found in memory.")
    if "X_tda_full" not in dir():
        raise ValueError("X_tda_full not found in memory.")
    X_tda_all = X_tda_full.copy()

perm_series = pd.Series(avg_perm).sort_values()

print("\n" + "=" * 80)
print("HOLDOUT EXPERIMENT (W=96): HAR-RV / vol-baseline / +IT / +TDA / +both")
print("=" * 80)

BEST_IT = [f for f in BEST_IT_96 if f in X_it_all.columns]

tda_candidates = [
    f for f in perm_series.index
    if (f in X_tda_all.columns) and (perm_series[f] < PERM_USEFUL_THRESHOLD)
]
BEST_TDA = tda_candidates[:TOP_TDA]

print("\nSelected IT features (W=96):")
for f in BEST_IT:
    print(f"  {f:<36} perm={perm_series.get(f, float('nan')):+.4f}")

print("\nSelected TDA features:")
for f in BEST_TDA:
    print(f"  {f:<36} perm={perm_series[f]:+.4f}")

if len(BEST_IT) == 0:
    raise ValueError("No IT features found for W=96.")
if len(BEST_TDA) == 0:
    raise ValueError("No useful TDA features survived the threshold.")

print(f"\nIT count:  {len(BEST_IT)}")
print(f"TDA count: {len(BEST_TDA)}")

split_ts = pd.Timestamp(SPLIT_DATE)
rows = []

for asset in ASSETS:
    r = ret[asset]

    X_har = build_har_features(r)

    for h, h_label in HORIZONS.items():
        vol_fwd = r.rolling(h).std().shift(-h) * ANN
        y_raw   = np.log(vol_fwd.clip(lower=1e-8)).dropna()

        Xi_it  = X_it_all[BEST_IT].copy()
        Xi_tda = X_tda_all[BEST_TDA].copy()

        idx = (
            X_vol.index
            .intersection(Xi_it.index)
            .intersection(Xi_tda.index)
            .intersection(X_har.index)
            .intersection(y_raw.index)
        )

        valid = (
            X_vol.loc[idx].notna().all(1)
            & Xi_it.loc[idx].notna().all(1)
            & Xi_tda.loc[idx].notna().all(1)
            & X_har.loc[idx].notna().all(1)
            & y_raw.loc[idx].notna()
        )
        idx = idx[valid]

        if len(idx) < 1000:
            continue

        tr = idx < (split_ts - pd.Timedelta(minutes=30 * PURGE_BARS))
        te = idx >= split_ts

        if tr.sum() < 480 or te.sum() < 100:
            continue

        Xv_tr  = X_vol.loc[idx[tr]];  Xv_te  = X_vol.loc[idx[te]]
        Xit_tr = Xi_it.loc[idx[tr]];  Xit_te = Xi_it.loc[idx[te]]
        Xtd_tr = Xi_tda.loc[idx[tr]]; Xtd_te = Xi_tda.loc[idx[te]]
        Xh_tr  = X_har.loc[idx[tr]];  Xh_te  = X_har.loc[idx[te]]

        ytr   = y_raw.loc[idx[tr]]
        yte   = y_raw.loc[idx[te]]
        naive = float(ytr.mean())

        m_har = _ridge()
        m_har.fit(Xh_tr, ytr)
        ph_te  = m_har.predict(Xh_te)
        r2_HAR = oos_r2(yte.values, ph_te, naive)
        mse_HAR = np.mean((yte.values - ph_te) ** 2)

        garch_ser = garch_forecasts.get((asset, h), pd.Series(dtype=float))
        garch_te  = garch_ser.reindex(idx[te]).values

        garch_valid = np.isfinite(garch_te)
        if garch_valid.sum() > 10:
            yte_g  = yte.values[garch_valid]
            pg_te  = garch_te[garch_valid]
            naive_g = float(ytr.mean())
            r2_GARCH  = oos_r2(yte_g, pg_te, naive_g)
            mse_GARCH = np.mean((yte_g - pg_te) ** 2)
            garch_te_full = np.where(garch_valid, garch_te, naive)
        else:
            r2_GARCH      = np.nan
            mse_GARCH     = np.nan
            garch_te_full = np.full(len(yte), naive)

        ck   = Xv_tr.corrwith(ytr).abs().replace([np.inf, -np.inf], np.nan).dropna()
        top3 = ck.nlargest(3).index.tolist()

        mb = _ridge()
        mb.fit(Xv_tr[top3], ytr)
        pb_tr = mb.predict(Xv_tr[top3])
        pb_te = mb.predict(Xv_te[top3])

        res_tr = ytr.values - pb_tr
        res_te = yte.values - pb_te

        mit = _ridge()
        mit.fit(Xit_tr, res_tr)
        pit_te  = mit.predict(Xit_te)
        pred_it = pb_te + pit_te

        mtda = _ridge()
        mtda.fit(Xtd_tr, res_tr)
        ptda_te  = mtda.predict(Xtd_te)
        pred_tda = pb_te + ptda_te

        Xc_tr = pd.concat([Xit_tr, Xtd_tr], axis=1)
        Xc_te = pd.concat([Xit_te, Xtd_te], axis=1)

        mct = _ridge()
        mct.fit(Xc_tr, res_tr)
        pct_te   = mct.predict(Xc_te)
        pred_all = pb_te + pct_te

        r2_A = oos_r2(yte.values, pb_te,    naive)
        r2_B = oos_r2(yte.values, pred_it,  naive)
        r2_C = oos_r2(yte.values, pred_tda, naive)
        r2_D = oos_r2(yte.values, pred_all, naive)

        rc_B = res_corr(res_te, pit_te)
        rc_C = res_corr(res_te, ptda_te)
        rc_D = res_corr(res_te, pct_te)

        mse_A = np.mean((yte.values - pb_te) ** 2)
        mse_B = np.mean((yte.values - pred_it) ** 2)
        mse_C = np.mean((yte.values - pred_tda) ** 2)
        mse_D = np.mean((yte.values - pred_all) ** 2)

        hr_B = hit_rate_vs_baseline(yte.values, pb_te, pred_it)
        hr_C = hit_rate_vs_baseline(yte.values, pb_te, pred_tda)
        hr_D = hit_rate_vs_baseline(yte.values, pb_te, pred_all)

        dB = (yte.values - pb_te)**2 - (yte.values - pred_it)**2
        dC = (yte.values - pb_te)**2 - (yte.values - pred_tda)**2
        dD = (yte.values - pb_te)**2 - (yte.values - pred_all)**2
        _, p_B = dm_test(dB)
        _, p_C = dm_test(dC)
        _, p_D = dm_test(dD)

        dD_vs_HAR = (yte.values - ph_te)**2 - (yte.values - pred_all)**2
        _, p_D_vs_HAR = dm_test(dD_vs_HAR)

        dD_vs_GARCH = (yte.values - garch_te_full)**2 - (yte.values - pred_all)**2
        _, p_D_vs_GARCH = dm_test(dD_vs_GARCH)

        rows.append({
            "asset":   asset,
            "horizon": h_label,
            "h":       h,

            "r2_HAR":   round(r2_HAR,   4),
            "r2_GARCH": round(r2_GARCH, 4) if np.isfinite(r2_GARCH) else np.nan,
            "r2_A":     round(r2_A,     4),
            "r2_B":     round(r2_B,     4),
            "r2_C":     round(r2_C,     4),
            "r2_D":     round(r2_D,     4),

            "dR2_B": round(r2_B - r2_A, 4),
            "dR2_C": round(r2_C - r2_A, 4),
            "dR2_D": round(r2_D - r2_A, 4),

            "dR2_D_vs_HAR":   round(r2_D - r2_HAR,   4),
            "dR2_D_vs_GARCH": round(r2_D - r2_GARCH, 4) if np.isfinite(r2_GARCH) else np.nan,

            "rc_B": round(rc_B, 4),
            "rc_C": round(rc_C, 4),
            "rc_D": round(rc_D, 4),

            "mse_ratio_B": round(mse_B / mse_A, 4) if mse_A > 0 else np.nan,
            "mse_ratio_C": round(mse_C / mse_A, 4) if mse_A > 0 else np.nan,
            "mse_ratio_D": round(mse_D / mse_A, 4) if mse_A > 0 else np.nan,

            "mse_ratio_D_vs_HAR":   round(mse_D / mse_HAR,   4) if mse_HAR   > 0 else np.nan,
            "mse_ratio_D_vs_GARCH": round(mse_D / mse_GARCH, 4) if (np.isfinite(r2_GARCH) and mse_GARCH > 0) else np.nan,

            "hit_B": round(hr_B, 4),
            "hit_C": round(hr_C, 4),
            "hit_D": round(hr_D, 4),

            "p_B": round(p_B, 4),
            "p_C": round(p_C, 4),
            "p_D": round(p_D, 4),
            "p_D_vs_HAR":   round(p_D_vs_HAR,   4),
            "p_D_vs_GARCH": round(p_D_vs_GARCH, 4),

            "sig_B":          sig(p_B),
            "sig_C":          sig(p_C),
            "sig_D":          sig(p_D),
            "sig_D_vs_HAR":   sig(p_D_vs_HAR),
            "sig_D_vs_GARCH": sig(p_D_vs_GARCH),

            "n_train": int(tr.sum()),
            "n_test":  int(te.sum()),
        })

df_best = pd.DataFrame(rows)
print(f"\nRows collected: {len(df_best)}")

if df_best.empty:
    raise ValueError("The result dataframe is empty.")

print("\n" + "=" * 115)
print("FINAL COMPARISON WITH AUTOMATICALLY SELECTED BEST FEATURES")
print("Target: logvol = log(realized_vol_fwd_h)")
print("HAR = HAR-RV | GARCH = GARCH(1,1) | A = vol-baseline | B = +IT | C = +TDA | D = +IT+TDA")
print("=" * 115)

print(
    f"\n  {'asset':<6} {'h':<4}"
    f" {'R²_HAR':>9} {'R²_GARCH':>10} {'R²_A':>9}"
    f" {'R²_B':>9} {'R²_C':>9} {'R²_D':>9}"
    f" {'ΔD/HAR':>9} {'ΔD/GARCH':>10}  best"
)
print("  " + "-" * 115)

prev = None
for _, row in df_best.sort_values(["asset", "h"]).iterrows():
    if prev and prev != row["asset"]:
        print("  " + "·" * 115)
    prev = row["asset"]

    vals = {
        "HAR":   row["r2_HAR"],
        "GARCH": row["r2_GARCH"] if np.isfinite(row["r2_GARCH"]) else -999,
        "A":     row["r2_A"],
        "B":     row["r2_B"],
        "C":     row["r2_C"],
        "D":     row["r2_D"],
    }
    best = max(vals, key=vals.get)

    dh = row["dR2_D_vs_HAR"]
    dg = row["dR2_D_vs_GARCH"]
    mh = "↑" if dh >  0.005 else ("↓" if dh < -0.005 else "~")
    mg = "↑" if (np.isfinite(dg) and dg >  0.005) else \
         ("↓" if (np.isfinite(dg) and dg < -0.005) else "~")

    garch_str = f"{row['r2_GARCH']:>+10.4f}" if np.isfinite(row["r2_GARCH"]) else f"{'N/A':>10}"
    dg_str    = f"{dg:>+9.4f}{mg}" if np.isfinite(dg) else f"{'N/A':>10}"

    print(
        f"  {row['asset']:<6} {row['horizon']:<4}"
        f" {row['r2_HAR']:>+9.4f}"
        f" {garch_str}"
        f" {row['r2_A']:>+9.4f}"
        f" {row['r2_B']:>+9.4f}"
        f" {row['r2_C']:>+9.4f}"
        f" {row['r2_D']:>+9.4f}"
        f" {row['dR2_D_vs_HAR']:>+8.4f}{mh}"
        f" {dg_str}"
        f"  {best}"
    )

print("\n" + "=" * 75)
print("SUMMARY")
print("=" * 75)

n = len(df_best)
n_garch = df_best["r2_GARCH"].notna().sum()

print(f"""
  Config              mean R²    mean ΔR² vs A   ΔR²>0
  -------------------------------------------------------
  HAR-RV              {df_best['r2_HAR'].mean():>+8.4f}    —
  GARCH(1,1)          {df_best['r2_GARCH'].mean():>+8.4f}    —         (N={n_garch}/{n} valid)
  A  (vol baseline)   {df_best['r2_A'].mean():>+8.4f}    —
  B  (+best IT)       {df_best['r2_B'].mean():>+8.4f}   {df_best['dR2_B'].mean():>+8.4f}   {(df_best['dR2_B']>0).sum():>2}/{n}
  C  (+best TDA)      {df_best['r2_C'].mean():>+8.4f}   {df_best['dR2_C'].mean():>+8.4f}   {(df_best['dR2_C']>0).sum():>2}/{n}
  D  (+both)          {df_best['r2_D'].mean():>+8.4f}   {df_best['dR2_D'].mean():>+8.4f}   {(df_best['dR2_D']>0).sum():>2}/{n}

  D vs HAR-RV:    mean ΔR²={df_best['dR2_D_vs_HAR'].mean():>+.4f}   D beats HAR:   {(df_best['dR2_D_vs_HAR']>0).sum()}/{n}
  D vs GARCH(1,1): mean ΔR²={df_best['dR2_D_vs_GARCH'].mean():>+.4f}   D beats GARCH: {(df_best['dR2_D_vs_GARCH']>0).sum()}/{n_garch}

  DM test (D vs HAR-RV):
    p<0.01: {(df_best['p_D_vs_HAR']<0.01).sum()}/{n}   p<0.05: {(df_best['p_D_vs_HAR']<0.05).sum()}/{n}   p<0.10: {(df_best['p_D_vs_HAR']<0.10).sum()}/{n}
  DM test (D vs GARCH):
    p<0.01: {(df_best['p_D_vs_GARCH']<0.01).sum()}/{n}   p<0.05: {(df_best['p_D_vs_GARCH']<0.05).sum()}/{n}   p<0.10: {(df_best['p_D_vs_GARCH']<0.10).sum()}/{n}
""")

print("Top-5 tasks by ΔR² (D vs HAR-RV):")
for _, row in df_best.nlargest(5, "dR2_D_vs_HAR").iterrows():
    print(
        f"  {row['asset']} h={row['horizon']}: "
        f"R²_HAR={row['r2_HAR']:+.4f}  "
        f"R²_GARCH={row['r2_GARCH']:+.4f}  "
        f"R²_D={row['r2_D']:+.4f}  "
        f"Δ_HAR={row['dR2_D_vs_HAR']:+.4f}  "
        f"p={row['p_D_vs_HAR']:.4f}{row['sig_D_vs_HAR']}"
    )

asset_summary = (
    df_best.groupby("asset", as_index=False)
    .agg(
        r2_HAR=("r2_HAR",   "mean"),
        r2_GARCH=("r2_GARCH","mean"),
        r2_A=("r2_A", "mean"),
        r2_B=("r2_B", "mean"),
        r2_C=("r2_C", "mean"),
        r2_D=("r2_D", "mean"),
        dR2_B=("dR2_B", "mean"),
        dR2_C=("dR2_C", "mean"),
        dR2_D=("dR2_D", "mean"),
        dR2_D_vs_HAR=("dR2_D_vs_HAR",   "mean"),
        dR2_D_vs_GARCH=("dR2_D_vs_GARCH","mean"),
        rc_B=("rc_B", "mean"),
        rc_C=("rc_C", "mean"),
        rc_D=("rc_D", "mean"),
        mse_ratio_B=("mse_ratio_B", "mean"),
        mse_ratio_C=("mse_ratio_C", "mean"),
        mse_ratio_D=("mse_ratio_D", "mean"),
        hit_B=("hit_B", "mean"),
        hit_C=("hit_C", "mean"),
        hit_D=("hit_D", "mean"),
    )
)

def choose_best_config(row):
    vals = {
        "HAR":   row["r2_HAR"],
        "GARCH": row["r2_GARCH"] if np.isfinite(row["r2_GARCH"]) else -999,
        "A":     row["r2_A"],
        "B":     row["r2_B"],
        "C":     row["r2_C"],
        "D":     row["r2_D"],
    }
    return max(vals, key=vals.get)

def choose_best_addon(row):
    vals = {"B": row["dR2_B"], "C": row["dR2_C"], "D": row["dR2_D"]}
    return max(vals, key=vals.get)

asset_summary["best_config"] = asset_summary.apply(choose_best_config, axis=1)
asset_summary["best_addon"]  = asset_summary.apply(choose_best_addon,  axis=1)
asset_summary = asset_summary.sort_values(
    ["r2_D","r2_C","r2_B","r2_A"], ascending=False
).reset_index(drop=True)

print("\n" + "=" * 115)
print("ASSET-LEVEL AGGREGATION (MEAN OVER HORIZONS)")
print("=" * 115)

print(
    f"\n  {'asset':<6}"
    f" {'R²_HAR':>9} {'R²_GARCH':>10} {'R²_A':>9}"
    f" {'R²_B':>9} {'R²_C':>9} {'R²_D':>9}"
    f" {'ΔD/HAR':>9} {'ΔD/GARCH':>10}"
    f" {'best_cfg':>10}"
)
print("  " + "-" * 115)

for _, row in asset_summary.iterrows():
    garch_str = f"{row['r2_GARCH']:>+10.4f}" if np.isfinite(row["r2_GARCH"]) else f"{'N/A':>10}"
    dg_str    = f"{row['dR2_D_vs_GARCH']:>+10.4f}" if np.isfinite(row["dR2_D_vs_GARCH"]) else f"{'N/A':>10}"
    print(
        f"  {row['asset']:<6}"
        f" {row['r2_HAR']:>+9.4f}"
        f" {garch_str}"
        f" {row['r2_A']:>+9.4f}"
        f" {row['r2_B']:>+9.4f}"
        f" {row['r2_C']:>+9.4f}"
        f" {row['r2_D']:>+9.4f}"
        f" {row['dR2_D_vs_HAR']:>+9.4f}"
        f" {dg_str}"
        f" {row['best_config']:>10}"
    )

print("\nBest configuration counts across assets:")
print(asset_summary["best_config"].value_counts().sort_index())

print("\nBuilding final plots...")

fig, axes = plt.subplots(2, 3, figsize=(21, 11))

asset_order = (
    df_best.groupby("asset")["r2_D"]
    .mean()
    .sort_values(ascending=False)
    .index.tolist()
)
h_order = ["1d","2d","4d","6d","7d","14d"]
x = np.arange(len(asset_order))
w = 0.14

ax = axes[0, 0]
for ci, (col, label, color) in enumerate([
    ("r2_HAR",   "HAR-RV",       "#795548"),
    ("r2_GARCH", "GARCH(1,1)",   "#9C27B0"),
    ("r2_A",     "A: Baseline",  "#9E9E9E"),
    ("r2_B",     "B: +IT",       "#2196F3"),
    ("r2_C",     "C: +TDA",      "#FF9800"),
    ("r2_D",     "D: +both",     "#4CAF50"),
]):
    vals = [df_best[df_best.asset == a][col].mean() for a in asset_order]
    ax.bar(x + (ci - 2.5) * w, vals, w * 0.9, label=label, color=color, alpha=0.85)

ax.axhline(0, color="black", lw=0.8)
ax.set_xticks(x)
ax.set_xticklabels(asset_order, fontsize=8)
ax.set_ylabel("OOS R²")
ax.set_title("OOS R² by asset — 6 configurations\n(logvol, all horizons)", fontsize=10)
ax.legend(fontsize=7)
ax.grid(alpha=0.2, axis="y")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax = axes[0, 1]
for col, label, color, ls in [
    ("r2_HAR",   "HAR-RV",      "#795548", ":"),
    ("r2_GARCH", "GARCH(1,1)",  "#9C27B0", ":"),
    ("r2_A",     "A: Baseline", "#9E9E9E", "-"),
    ("r2_B",     "B: +IT",      "#2196F3", "-"),
    ("r2_C",     "C: +TDA",     "#FF9800", "--"),
    ("r2_D",     "D: +both",    "#4CAF50", "-"),
]:
    vals = [df_best[df_best.horizon == h][col].mean() for h in h_order]
    ax.plot(h_order, vals, "o" + ls, color=color, lw=2, label=label, ms=6)

ax.axhline(0, color="black", lw=0.5, ls=":")
ax.set_ylabel("OOS R²")
ax.set_xlabel("Forecast horizon")
ax.set_title("OOS R² by horizon\n(mean over assets)", fontsize=10)
ax.legend(fontsize=7)
ax.grid(alpha=0.2)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax = axes[0, 2]
vals_har = [df_best[df_best.asset==a]["dR2_D_vs_HAR"].mean() for a in asset_order]
colors_bar = ["#4CAF50" if v >= 0 else "#F44336" for v in vals_har]
ax.bar(np.arange(len(asset_order)), vals_har, color=colors_bar, alpha=0.85)
ax.axhline(0, color="black", lw=0.8)
ax.set_xticks(np.arange(len(asset_order)))
ax.set_xticklabels(asset_order, fontsize=9)
ax.set_ylabel("ΔR² (D minus HAR-RV)")
ax.set_title("Configuration D vs HAR-RV\n(mean over horizons)", fontsize=10)
ax.grid(alpha=0.2, axis="y")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax = axes[1, 0]
x2 = np.arange(len(asset_order))
w2 = 0.28
for ci, (col, label, color) in enumerate([
    ("dR2_B", "B: +IT",   "#2196F3"),
    ("dR2_C", "C: +TDA",  "#FF9800"),
    ("dR2_D", "D: +both", "#4CAF50"),
]):
    vals = [df_best[df_best.asset==a][col].mean() for a in asset_order]
    ax.bar(x2 + (ci-1)*w2, vals, w2*0.9, label=label, color=color, alpha=0.85)

ax.axhline(0, color="black", lw=0.8)
ax.set_xticks(x2)
ax.set_xticklabels(asset_order, fontsize=9)
ax.set_ylabel("ΔR² relative to baseline A")
ax.set_title("Incremental ΔR² over vol baseline\n(logvol, all horizons)", fontsize=10)
ax.legend(fontsize=8)
ax.grid(alpha=0.2, axis="y")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax = axes[1, 1]
pivot = df_best.pivot_table(
    index="asset", columns="horizon",
    values="mse_ratio_D", aggfunc="mean"
).reindex(asset_order)[h_order]

im = ax.imshow(pivot.values, cmap="RdYlGn_r", vmin=0.85, vmax=1.15, aspect="auto")
ax.set_xticks(range(len(h_order)));    ax.set_xticklabels(h_order)
ax.set_yticks(range(len(asset_order))); ax.set_yticklabels(asset_order)
for i in range(len(asset_order)):
    for j in range(len(h_order)):
        val = pivot.values[i, j]
        if np.isfinite(val):
            col = "white" if abs(val-1) > 0.07 else "black"
            mark = "✓" if val < 1.0 else ""
            ax.text(j, i, f"{val:.3f}{mark}", ha="center", va="center",
                    fontsize=8, color=col)
plt.colorbar(im, ax=ax, label="MSE ratio (D / baseline A)\n<1.0 = better ✓")
ax.set_title("MSE ratio: D vs vol baseline\n(green < 1 = better)", fontsize=10)

ax = axes[1, 2]
pivot2 = df_best.pivot_table(
    index="asset", columns="horizon",
    values="mse_ratio_D_vs_HAR", aggfunc="mean"
).reindex(asset_order)[h_order]

im2 = ax.imshow(pivot2.values, cmap="RdYlGn_r", vmin=0.85, vmax=1.15, aspect="auto")
ax.set_xticks(range(len(h_order)));    ax.set_xticklabels(h_order)
ax.set_yticks(range(len(asset_order))); ax.set_yticklabels(asset_order)
for i in range(len(asset_order)):
    for j in range(len(h_order)):
        val = pivot2.values[i, j]
        if np.isfinite(val):
            col = "white" if abs(val-1) > 0.07 else "black"
            mark = "✓" if val < 1.0 else ""
            ax.text(j, i, f"{val:.3f}{mark}", ha="center", va="center",
                    fontsize=8, color=col)
plt.colorbar(im2, ax=ax, label="MSE ratio (D / HAR-RV)\n<1.0 = better ✓")
ax.set_title("MSE ratio: D vs HAR-RV\n(green < 1 = better)", fontsize=10)

plt.suptitle(
    "Final comparison (IT W=96): HAR-RV / vol-baseline / +IT / +TDA / +both\n"
    f"Target: logvol = log(realized_vol_fwd) | holdout {SPLIT_DATE} →",
    fontsize=12, y=1.01
)
plt.tight_layout()
plt.savefig("fig_final_comparison_W96.png", bbox_inches="tight", dpi=150)
plt.savefig("fig_final_comparison_W96.pdf", bbox_inches="tight", dpi=150)
plt.show()
print("Saved: fig_final_comparison_W96.png / .pdf")

final_comparison_W96 = df_best.copy()
best_feature_sets_W96 = {"BEST_IT": BEST_IT, "BEST_TDA": BEST_TDA}
print(f"\nfinal_comparison_W96 saved in memory ({len(df_best)} rows)")
print("best_feature_sets_W96 saved in memory")

Building GARCH(1,1) forecasts for all assets and horizons...
  BTC h=1d: 2977 test points
  BTC h=2d: 2977 test points
  BTC h=4d: 2977 test points
  BTC h=6d: 2977 test points
  BTC h=7d: 2977 test points
  BTC h=14d: 2977 test points
  ETH h=1d: 2977 test points
  ETH h=2d: 2977 test points
  ETH h=4d: 2977 test points
  ETH h=6d: 2977 test points
  ETH h=7d: 2977 test points
  ETH h=14d: 2977 test points
  SOL h=1d: 2977 test points
  SOL h=2d: 2977 test points
  SOL h=4d: 2977 test points
  SOL h=6d: 2977 test points
  SOL h=7d: 2977 test points
  SOL h=14d: 2977 test points
  BNB h=1d: 2977 test points
  BNB h=2d: 2977 test points
  BNB h=4d: 2977 test points
  BNB h=6d: 2977 test points
  BNB h=7d: 2977 test points
  BNB h=14d: 2977 test points
  XRP h=1d: 2977 test points
  XRP h=2d: 2977 test points
  XRP h=4d: 2977 test points
  XRP h=6d: 2977 test points
  XRP h=7d: 2977 test points
  XRP h=14d: 2977 test points
  DOGE h=1d: 2977 test points
  DOGE h=2d: 2977 test points
  DO

## 8. Stress classification: window sensitivity

In [ ]:
df_sw = stress_window_sensitivity_results.copy()
WINDOWS_TO_TEST = sorted(
    df_sw["window"].str.replace("W=","").astype(int).unique().tolist()
)
QUANTILES_list = sorted(df_sw["q"].unique().tolist())
ASSETS_order   = ["BTC","ETH","SOL","BNB","XRP","DOGE","ADA","AVAX"]
HORIZONS_order = ["1d","2d","4d"]

for w in WINDOWS_TO_TEST:
    wl  = f"W={w}"
    sub_w = df_sw[df_sw.window == wl]
    n   = len(sub_w)

    print("\n" + "=" * 100)
    print(f"WINDOW {wl}  (~{w/48:.1f} days) — {n} tasks")
    print("A = vol_now | B = +IT | C = +TDA | D = +IT+TDA")
    print("=" * 100)

    print(f"\n  By threshold (mean over assets and horizons):")
    print(f"  {'Q':>5} {'pos%':>5}"
          f" {'ROC_A':>7} {'ROC_B':>7} {'ROC_C':>7} {'ROC_D':>7}"
          f" {'ΔROC_B':>8} {'ΔROC_C':>8} {'ΔROC_D':>8}"
          f" {'D>A':>6} {'ΔPR_D':>8}")
    print("  " + "-" * 85)

    for q in QUANTILES_list:
        sub = sub_w[sub_w.q == q]
        if sub.empty: continue
        beats = (sub["droc_D"] > 0.01).sum()
        print(f"  Q{q:.0%}"
              f" {sub['pos_te'].mean():>4.0%}"
              f" {sub['roc_A'].mean():>7.4f}"
              f" {sub['roc_B'].mean():>7.4f}"
              f" {sub['roc_C'].mean():>7.4f}"
              f" {sub['roc_D'].mean():>7.4f}"
              f" {sub['droc_B'].mean():>+8.4f}"
              f" {sub['droc_C'].mean():>+8.4f}"
              f" {sub['droc_D'].mean():>+8.4f}"
              f" {beats:>4}/{len(sub)}"
              f" {sub['dpr_D'].mean():>+8.4f}")

    print(f"\n  By asset at Q80 (mean over horizons):")
    print(f"  {'asset':<6} {'pos%':>5}"
          f" {'ROC_A':>7} {'ROC_B':>7} {'ROC_C':>7} {'ROC_D':>7}"
          f" {'ΔROC_B':>8} {'ΔROC_C':>8} {'ΔROC_D':>8}"
          f" {'ΔPR_D':>8}  best")
    print("  " + "-" * 90)

    sub80 = sub_w[sub_w.q == 0.80]
    for asset in ASSETS_order:
        g = sub80[sub80.asset == asset]
        if g.empty: continue
        vals = {"A": g["roc_A"].mean(), "B": g["roc_B"].mean(),
                "C": g["roc_C"].mean(), "D": g["roc_D"].mean()}
        best = max(vals, key=vals.get)
        print(f"  {asset:<6}"
              f" {g['pos_te'].mean():>4.0%}"
              f" {g['roc_A'].mean():>7.4f}"
              f" {g['roc_B'].mean():>7.4f}"
              f" {g['roc_C'].mean():>7.4f}"
              f" {g['roc_D'].mean():>7.4f}"
              f" {g['droc_B'].mean():>+8.4f}"
              f" {g['droc_C'].mean():>+8.4f}"
              f" {g['droc_D'].mean():>+8.4f}"
              f" {g['dpr_D'].mean():>+8.4f}  -> {best}")

    print(f"\n  By horizon at Q80 (mean over assets):")
    print(f"  {'h':<4}"
          f" {'ROC_A':>7} {'ROC_B':>7} {'ROC_C':>7} {'ROC_D':>7}"
          f" {'ΔROC_B':>8} {'ΔROC_C':>8} {'ΔROC_D':>8}")
    print("  " + "-" * 68)

    for h_label in HORIZONS_order:
        g = sub80[sub80.horizon == h_label]
        if g.empty: continue
        print(f"  {h_label:<4}"
              f" {g['roc_A'].mean():>7.4f}"
              f" {g['roc_B'].mean():>7.4f}"
              f" {g['roc_C'].mean():>7.4f}"
              f" {g['roc_D'].mean():>7.4f}"
              f" {g['droc_B'].mean():>+8.4f}"
              f" {g['droc_C'].mean():>+8.4f}"
              f" {g['droc_D'].mean():>+8.4f}")

    print(f"\n  Overall ({n} tasks):"
          f"  ΔROC_B={sub_w['droc_B'].mean():>+.4f}"
          f"  ΔROC_C={sub_w['droc_C'].mean():>+.4f}"
          f"  ΔROC_D={sub_w['droc_D'].mean():>+.4f}"
          f"  ΔPR_D={sub_w['dpr_D'].mean():>+.4f}"
          f"  D>A: {(sub_w['droc_D']>0.01).sum()}/{n}")

print("\n" + "=" * 80)
print("CROSS-WINDOW SUMMARY")
print("=" * 80)

print(f"\n  {'Window':<10} {'days':>5}"
      f" {'ΔROC_B':>8} {'ΔROC_C':>8} {'ΔROC_D':>8}"
      f" {'ΔPR_B':>8} {'ΔPR_C':>8} {'ΔPR_D':>8}"
      f" {'D>A':>8}")
print("  " + "-" * 78)

for w in WINDOWS_TO_TEST:
    wl  = f"W={w}"
    sub = df_sw[df_sw.window == wl]
    if sub.empty: continue
    n = len(sub)
    beats = (sub["droc_D"] > 0.01).sum()
    print(f"  {wl:<10} {w/48:>5.1f}"
          f" {sub['droc_B'].mean():>+8.4f}"
          f" {sub['droc_C'].mean():>+8.4f}"
          f" {sub['droc_D'].mean():>+8.4f}"
          f" {sub['dpr_B'].mean():>+8.4f}"
          f" {sub['dpr_C'].mean():>+8.4f}"
          f" {sub['dpr_D'].mean():>+8.4f}"
          f" {beats:>5}/{n}")

print(f"\n  Asset-level ΔROC_D at Q80:")
print(f"  {'Asset':<6}", end="")
for w in WINDOWS_TO_TEST:
    print(f"  {'W='+str(w):>10}", end="")
print()
print("  " + "-" * (6 + 12*len(WINDOWS_TO_TEST)))

for asset in ASSETS_order:
    print(f"  {asset:<6}", end="")
    for w in WINDOWS_TO_TEST:
        wl  = f"W={w}"
        sub = df_sw[(df_sw.window==wl) &
                    (df_sw.asset==asset) &
                    (df_sw.q==0.80)]
        val = sub["droc_D"].mean() if not sub.empty else float("nan")
        if val == val:
            print(f"  {val:>+10.4f}", end="")
        else:
            print(f"  {'N/A':>10}", end="")
    print()

print(f"\n  TDA invariance check (ΔROC_C fixed W=192):")
print(f"  {'Window':<10} {'ΔROC_C':>8} {'ΔPR_C':>8}")
for w in WINDOWS_TO_TEST:
    wl  = f"W={w}"
    sub = df_sw[df_sw.window == wl]
    if sub.empty: continue
    print(f"  {wl:<10} {sub['droc_C'].mean():>+8.4f}"
          f" {sub['dpr_C'].mean():>+8.4f}")


WINDOW W=96  (~2.0 days) — 80 tasks
A = vol_now | B = +IT | C = +TDA | D = +IT+TDA

  By threshold (mean over assets and horizons):
      Q  pos%   ROC_A   ROC_B   ROC_C   ROC_D   ΔROC_B   ΔROC_C   ΔROC_D    D>A    ΔPR_D
  -------------------------------------------------------------------------------------
  Q60%  46%  0.6840  0.6952  0.7108  0.7183  +0.0112  +0.0268  +0.0344   18/23  +0.0470
  Q70%  32%  0.6832  0.7008  0.6980  0.7111  +0.0176  +0.0148  +0.0280   17/22  +0.0677
  Q80%  17%  0.6608  0.7019  0.6837  0.7067  +0.0411  +0.0229  +0.0459   15/21  +0.1080
  Q90%   5%  0.6250  0.6622  0.6193  0.6370  +0.0371  -0.0058  +0.0120    5/14  +0.0015

  By asset at Q80 (mean over horizons):
  asset   pos%   ROC_A   ROC_B   ROC_C   ROC_D   ΔROC_B   ΔROC_C   ΔROC_D    ΔPR_D  best
  ------------------------------------------------------------------------------------------
  BTC      1%  0.3506  0.2805  0.3121  0.2711  -0.0702  -0.0385  -0.0796  -0.0006  -> A
  ETH     28%  0.5895  0.60